In [1]:
import gc
import torch
import shutil
import os
import glob
from pathlib import Path

# 1. Clear GPU Memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory Cleared. Currently allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

# 2. Run Garbage Collection
gc.collect()

# 3. Redirect HF cache to a local folder instead of Google Drive
HF_CACHE_DIR = os.path.join(os.getcwd(), 'hf_cache')
os.environ['HF_HOME'] = HF_CACHE_DIR
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# 4. Clean up local disk (Temporary results directories)
for results_dir in glob.glob('./results_*'):
    if os.path.exists(results_dir):
        print(f"Removing temporary directory: {results_dir}")
        shutil.rmtree(results_dir)

if os.path.exists('./runs'):
    shutil.rmtree('./runs')

print(f"Memory and local disk cleanup complete. HF Cache redirected to: {os.environ['HF_HOME']}")

GPU Memory Cleared. Currently allocated: 0.00 MB
Memory and local disk cleanup complete. HF Cache redirected to: d:\Nepali sentiment\hf_cache


# Devnagiri Text — Sentiment Classification (XLM-R / MuRIL / mBERT)

This notebook fine-tunes and compares **three** multilingual transformer models — `xlm-roberta-base`, `google/muril-base-cased`, and `bert-base-multilingual-cased` — on the `English_Translation` column for **3-class sentiment classification** (Positive / Neutral / Negative).




### One pipeline, three independent runs
All three models are trained through **one shared function**, `run_sentiment_pipeline(model_key, model_name)`. Each call is fully self-contained (own tokenizer, own model, own trainer, own save directory) — so you run whichever model you need, in any order, without the others being trained first:
```python
results['XLM-R'] = run_sentiment_pipeline('xlmr',  'xlm-roberta-base')
results['MuRIL'] = run_sentiment_pipeline('muril', 'google/muril-base-cased')
results['mBERT'] = run_sentiment_pipeline('mbert', 'bert-base-multilingual-cased')
```
```
Dataset  : Combined_Common_File.xlsx
Input col: input_text (English_Translation column, mapped from source file)
Label col: sentiment  (Positive / Neutral / Negative)
```

## 0. Mount Drive & GPU Check

In [2]:
# Local-device notebook: Colab Drive mount is not required.
print('Local notebook environment active; Google Drive mount skipped.')

Local notebook environment active; Google Drive mount skipped.


In [3]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected — training will be very slow!")

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Imports

In [4]:
!pip install -q evaluate


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Install missing libraries
!pip install -q evaluate

import os
import gc
import shutil
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import evaluate
import warnings
warnings.filterwarnings('ignore')

print("All imports OK")


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


All imports OK


In [6]:
!pip install -q optuna


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

In [7]:
from huggingface_hub import login

login()

In [8]:
# ── Paths ─────────────────────────────────────────────────────────────────────

# Local data source
TRAIN_PATH = r"D:\Nepali sentiment\train_devanagari.xlsx"
VAL_PATH = r"D:\Nepali sentiment\val_devanagari.xlsx"
TEST_PATH = r"D:\Nepali sentiment\test_devanagari.xlsx"

# ============================================================
# D: DRIVE STORAGE
# ============================================================

D_ROOT = r'D:\Nepali sentiment'

# Main directories
MODEL_ROOT = os.path.join(D_ROOT, 'models')
TOKENIZER_ROOT = os.path.join(D_ROOT, 'tokenizers')
RESULTS_ROOT = os.path.join(D_ROOT, 'results')
TRAINING_ROOT = os.path.join(D_ROOT, 'training_checkpoints')

# Hugging Face cache
HF_ROOT = os.path.join(D_ROOT, 'hf_cache')

os.environ['HF_HOME'] = HF_ROOT
os.environ['HF_HUB_CACHE'] = os.path.join(HF_ROOT, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(HF_ROOT, 'transformers')

# Try a Hub login only if a valid token is available locally.
# If the environment is offline or the token cannot be validated, we still continue
# # with the local cache and already downloaded model files.
# try:
#     from huggingface_hub import login
#     hf_token = os.environ.get('HF_TOKEN')
#     if hf_token:
#         login(token=hf_token, add_to_git_credential=False)
#         print('Hugging Face login successful using HF_TOKEN.')
#     else:
#         print('HF_TOKEN not found; continuing in offline/cache mode.')
# except Exception as exc:
#     print(f'Hugging Face login skipped: {exc}')
#     print('Continuing with local cache / already-downloaded models.')

# Create all directories
for folder in [
    D_ROOT,
    MODEL_ROOT,
    TOKENIZER_ROOT,
    RESULTS_ROOT,
    TRAINING_ROOT,
    HF_ROOT,
    os.environ['HF_HUB_CACHE'],
    os.environ['TRANSFORMERS_CACHE'],
]:
    os.makedirs(folder, exist_ok=True)

print("=" * 70)
print("D: DRIVE STORAGE CONFIGURED")
print("=" * 70)
print(f"Root directory       : {D_ROOT}")
print(f"Models               : {MODEL_ROOT}")
print(f"Tokenizers           : {TOKENIZER_ROOT}")
print(f"Results              : {RESULTS_ROOT}")
print(f"Checkpoints          : {TRAINING_ROOT}")
print(f"HuggingFace cache    : {HF_ROOT}")
print("=" * 70)


# ── Label mapping ─────────────────────────────────────────────────────────────

# Paper (Thapa et al. 2023) uses Positive / Neutral / Negative

SENT_MAP = {
    'Positive': 0,
    'Neutral': 1,
    'Negative': 2,
}

ID2LABEL = {v: k for k, v in SENT_MAP.items()}
N_LABELS = len(SENT_MAP)


# ── Shared training hyper-parameters ──────────────────────────────────────────

MAX_LEN             = 256
SEED                = 42
EPOCHS              = 5
BATCH_TRAIN         = 16
BATCH_EVAL          = 32
LR                  = 1e-5
WEIGHT_DECAY        = 0.15
WARMUP_RATIO        = 0.1
GRAD_ACCUM          = 2
EARLY_STOP_PATIENCE = 2
SAVE_TOTAL_LIMIT    = 1


# ── The 3 models being compared ──────────────────────────────────────────────

MODEL_CONFIGS = [
    {
        'key': 'xlmr',
        'label': 'XLM-RoBERTa',
        'checkpoint': 'xlm-roberta-base',

        # Final trained model → D:
        'save_dir': os.path.join(
            MODEL_ROOT,
            'xlm-roberta-base_model'
        ),

        # Tokenizer → D:
        'tok_dir': os.path.join(
            TOKENIZER_ROOT,
            'xlm-roberta-base_tokenizer'
        ),

        # Trainer checkpoints/results → D:
        'output_dir': os.path.join(
            TRAINING_ROOT,
            'xlmr'
        ),
    },

    {
        'key': 'mbert',
        'label': 'mBERT',
        'checkpoint': 'bert-base-multilingual-cased',

        'save_dir': os.path.join(
            MODEL_ROOT,
            'mbert_sentiment_model'
        ),

        'tok_dir': os.path.join(
            TOKENIZER_ROOT,
            'mbert_sentiment_tokenizer'
        ),

        'output_dir': os.path.join(
            TRAINING_ROOT,
            'mbert'
        ),
    },

    {
        'key': 'muril',
        'label': 'MuRIL',
        'checkpoint': 'google/muril-base-cased',

        'save_dir': os.path.join(
            MODEL_ROOT,
            'muril_sentiment_model'
        ),

        'tok_dir': os.path.join(
            TOKENIZER_ROOT,
            'muril_sentiment_tokenizer'
        ),

        'output_dir': os.path.join(
            TRAINING_ROOT,
            'muril'
        ),
    },
]


print("\nModels configured:")
for c in MODEL_CONFIGS:
    print(
        f"  - {c['label']:<15} → {c['checkpoint']}"
    )
    print(
        f"      Model     : {c['save_dir']}"
    )
    print(
        f"      Tokenizer : {c['tok_dir']}"
    )
    print(
        f"      Training  : {c['output_dir']}"
    )

D: DRIVE STORAGE CONFIGURED
Root directory       : D:\Nepali sentiment
Models               : D:\Nepali sentiment\models
Tokenizers           : D:\Nepali sentiment\tokenizers
Results              : D:\Nepali sentiment\results
Checkpoints          : D:\Nepali sentiment\training_checkpoints
HuggingFace cache    : D:\Nepali sentiment\hf_cache

Models configured:
  - XLM-RoBERTa     → xlm-roberta-base
      Model     : D:\Nepali sentiment\models\xlm-roberta-base_model
      Tokenizer : D:\Nepali sentiment\tokenizers\xlm-roberta-base_tokenizer
      Training  : D:\Nepali sentiment\training_checkpoints\xlmr
  - mBERT           → bert-base-multilingual-cased
      Model     : D:\Nepali sentiment\models\mbert_sentiment_model
      Tokenizer : D:\Nepali sentiment\tokenizers\mbert_sentiment_tokenizer
      Training  : D:\Nepali sentiment\training_checkpoints\mbert
  - MuRIL           → google/muril-base-cased
      Model     : D:\Nepali sentiment\models\muril_sentiment_model
      Tokenizer : D:

```markdown
## 2.1. Data Consolidation
Combining multiple Excel sources into a single dataset for training and validation.
```

## 3. Data Loading & Preprocessing

In [9]:
import pandas as pd
import os


def prepare_dataset(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find file at {path}.")

    df = pd.read_excel(path) if path.endswith(('.xlsx', '.xls')) else pd.read_csv(path)

    target_text_col = 'Nepali_Translation'
    target_label_col = 'Sentiment'

    if target_text_col in df.columns and target_label_col in df.columns:
        df = df[[target_text_col, target_label_col]].copy()
    else:
        df = df.iloc[:, [3, 2]].copy()

    df.columns = ['input_text', 'sentiment_raw']
    df = df.dropna().reset_index(drop=True)

    df['sentiment_raw'] = (
        df['sentiment_raw']
        .astype(str)
        .str.strip()
        .str.capitalize()
        .replace({'Postive': 'Positive', 'Netural': 'Neutral'})
    )

    df['label'] = df['sentiment_raw'].map(SENT_MAP)
    df = df[df['label'].notna()].reset_index(drop=True)
    df['label'] = df['label'].astype(int)

    # Remove duplicate texts within each split before comparing splits.
    return df.drop_duplicates(subset=['input_text'], keep='first').reset_index(drop=True)


# ── Load datasets ──────────────────────────────────────────────
print("Loading specific datasets...")
train_df = prepare_dataset(TRAIN_PATH)
val_df   = prepare_dataset(VAL_PATH)
test_df  = prepare_dataset(TEST_PATH)

print("\nBefore cross-split dedup:")
print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── Cross-split deduplication ──────────────────────────────────
# Priority: train > val > test.
train_texts = set(train_df['input_text'].astype(str))
val_df = val_df[
    ~val_df['input_text'].astype(str).isin(train_texts)
].reset_index(drop=True)

train_val_texts = train_texts | set(val_df['input_text'].astype(str))
test_df = test_df[
    ~test_df['input_text'].astype(str).isin(train_val_texts)
].reset_index(drop=True)

print("\nAfter cross-split dedup:")
print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── Verify zero leakage ────────────────────────────────────────
t_set = set(train_df['input_text'].astype(str))
v_set = set(val_df['input_text'].astype(str))
e_set = set(test_df['input_text'].astype(str))
leakage = {
    'train_val': len(t_set & v_set),
    'train_test': len(t_set & e_set),
    'val_test': len(v_set & e_set),
}
print(f"  Leakage check: {leakage}")
assert all(value == 0 for value in leakage.values()), "Leakage still present!"
print("  ✅ Zero leakage confirmed.\n")

print("Successfully loaded datasets with zero cross-split leakage.")

Loading specific datasets...

Before cross-split dedup:
  Train: 8549 | Val: 1070 | Test: 1069

After cross-split dedup:
  Train: 8549 | Val: 1069 | Test: 1069
  Leakage check: {'train_val': 0, 'train_test': 0, 'val_test': 0}
  ✅ Zero leakage confirmed.

Successfully loaded datasets with zero cross-split leakage.


## 4. Train / Validation / Test Split

Stratified 80/10/10 split on the label column — computed once and shared by every model so the comparison across XLM-R / MuRIL / mBERT is apples-to-apples.

In [10]:
# The datasets were already loaded separately above by prepare_dataset().
# Do not split an undefined combined dataframe here.
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Safety check: if the same raw text appears across splits, report it.
train_texts = set(train_df['input_text'].astype(str))
val_texts = set(val_df['input_text'].astype(str))
test_texts = set(test_df['input_text'].astype(str))
leakage = {
    'train_val': len(train_texts & val_texts),
    'train_test': len(train_texts & test_texts),
    'val_test': len(val_texts & test_texts),
}

print(f"Train : {len(train_df)} rows")
print(f"Val   : {len(val_df)} rows")
print(f"Test  : {len(test_df)} rows")
print(f"Cross-split overlap after deduplication: {leakage}")
print("\nTrain distribution:")
print(train_df['sentiment_raw'].value_counts())
print("\nVal distribution:")
print(val_df['sentiment_raw'].value_counts())
print("\nTest distribution:")
print(test_df['sentiment_raw'].value_counts())

Train : 8549 rows
Val   : 1069 rows
Test  : 1069 rows
Cross-split overlap after deduplication: {'train_val': 0, 'train_test': 0, 'val_test': 0}

Train distribution:
sentiment_raw
Neutral     2911
Negative    2870
Positive    2768
Name: count, dtype: int64

Val distribution:
sentiment_raw
Neutral     364
Negative    360
Positive    345
Name: count, dtype: int64

Test distribution:
sentiment_raw
Neutral     365
Negative    359
Positive    345
Name: count, dtype: int64


In [11]:
train_df.head(3), val_df.head(3), test_df.head(3)

(                                          input_text sentiment_raw  label
 0  यस्तो सालो मिडिया मादरचोद त्यत्तिकै आइमाई बोल्...      Negative      2
 1  सोनिकाको अन्तर्वार्ता साह्रै मन पर्छ उहाँको बो...      Positive      0
 2                        थुक्क माचिक्ने लोकतन्त्र 😠😟      Negative      2,
                                           input_text sentiment_raw  label
 0         इन्टरभ्युको लागि हार्दिक आभार टिकाराम सर 🙏      Positive      0
 1              के गर्नु बुढी कमजोर भएपछि दुःख पाइन्छ      Negative      2
 2  यसो त्यो राँडीको किन इन्टरभ्यु लिएर आएको होला ...      Negative      2,
                                           input_text sentiment_raw  label
 0            वास्तवमै सही व्यक्ति दिनेश शर्मा रेग्मी      Positive      0
 1  कुनै हिरोहरूमा मलाई सबैभन्दा मन पर्ने दिनेश सा...      Positive      0
 2  अमित र प्रितीको आजको प्रस्तुति निकै राम्रो थिय...      Positive      0)

## 5. Dataset Class

Tokenization is model-specific (each model has its own tokenizer/vocab), so it happens inside `run_sentiment_pipeline` below rather than as a separate global step — but the `Dataset` wrapper itself is identical for every model.

In [12]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings: dict, labels: list):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx: int) -> dict:
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self) -> int:
        return len(self.labels)

print("SentimentDataset defined.")

SentimentDataset defined.


## 6. Class Weights

Computed once from `train_df` and reused by every model — the label distribution doesn't change between models, only the tokenizer/architecture does.

In [13]:
cw = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].tolist(),
)
class_weights = torch.tensor(cw, dtype=torch.float)
print(f"Class weights (Pos / Neu / Neg): {class_weights.numpy().round(4)}")

Class weights (Pos / Neu / Neg): [1.0295 0.9789 0.9929]


## 7. Custom Trainer with Class-Weighted Loss

We override `compute_loss` to pass `class_weights` to `CrossEntropyLoss`, giving heavier penalty to misclassified minority examples. Same trainer class used for all three models.

In [14]:
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Calculate standard cross-entropy loss without reduction
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        # Get the probability of the true class
        pt = torch.exp(-ce_loss)
        # Apply the focal loss modulating factor
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        # Apply class weights (alpha) if provided
        if self.alpha is not None:
            alpha_weights = self.alpha.to(targets.device)[targets]
            focal_loss = alpha_weights * focal_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

class EpochLossPrinter(TrainerCallback):
    """Prints training and validation loss at the end of each epoch."""
    
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        
        # Get training loss from the current epoch
        train_loss = None
        val_loss = None
        val_f1 = None
        
        for log in state.log_history:
            if "loss" in log and "eval_loss" not in log:
                if abs(log.get("epoch", 0) - epoch) < 0.01:
                    train_loss = log["loss"]
            if "eval_loss" in log:
                if abs(log.get("epoch", 0) - epoch) < 0.01:
                    val_loss = log["eval_loss"]
                    val_f1 = log.get("eval_macro_f1", None)
        
        print(f"\n  📊 Epoch {epoch}/{args.num_train_epochs}")
        if train_loss is not None:
            print(f"     Train Loss : {train_loss:.4f}")
        if val_loss is not None:
            print(f"     Val Loss   : {val_loss:.4f}")
        if val_f1 is not None:
            print(f"     Val F1     : {val_f1:.4f}")
        
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = FocalLoss(alpha=class_weights, gamma=2.0)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        # Compute focal loss instead of cross_entropy
        loss = self.criterion(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [15]:
class CustomSentimentHead(nn.Module):
    def __init__(self, hidden_size, num_labels):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(0.2) for _ in range(5)])
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, pooled_output):
        # Multi-Sample Dropout: Average logits from 5 different dropout masks
        logits_list = [self.classifier(dropout(pooled_output)) for dropout in self.dropouts]
        return torch.stack(logits_list, dim=0).mean(dim=0)

class CustomSentimentModel(nn.Module):
    def __init__(self, checkpoint, num_labels, id2label, label2id):
        super().__init__()
        from transformers import AutoModel
        self.num_labels = num_labels
        self.id2label = id2label
        self.label2id = label2id

        # Load backbone
        self.backbone = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.backbone.config.hidden_size

        # Custom Head
        self.classifier = CustomSentimentHead(hidden_size, num_labels)

    def gradient_checkpointing_enable(self, **kwargs):
        """Delegate gradient checkpointing to the backbone transformer."""
        if hasattr(self.backbone, "gradient_checkpointing_enable"):
            self.backbone.gradient_checkpointing_enable(**kwargs)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, **kwargs)

        # Use pooled output (usually [CLS] token representation)
        pooled_output = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state[:, 0]

        logits = self.classifier(pooled_output)

        loss = None
        # Note: Loss calculation is handled by the WeightedTrainer using FocalLoss

        from transformers.modeling_outputs import SequenceClassifierOutput
        return SequenceClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

## 8. Metrics

We track **accuracy** and **macro-F1** during training, and produce a full `classification_report` at the end — same metrics reported in Thapa et al. (2023).

In [16]:
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        )),
        "micro_f1": float(f1_score(
            labels,
            predictions,
            average="micro",
            zero_division=0,
        )),
        "weighted_f1": float(f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        )),
    }

In [17]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def plot_multiclass_roc(all_labels, all_probs, label_names, title='ROC Curve'):
    """
    Plots a multi-class ROC curve using the One-vs-Rest strategy.
    """
    n_classes = len(label_names)
    # Binarize labels for multi-class ROC
    y_test_bin = label_binarize(all_labels, classes=range(n_classes))

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    plt.figure(figsize=(8, 6))
    colors = ['#4C72B0', '#55A868', '#C44E52']

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

        plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
                 label=f'ROC {label_names[i]} (area = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

## 9. Pipeline Helpers

Two small utilities the training function relies on:
- `get_precision_kwargs()` checks what the *current* GPU actually supports instead of hardcoding `bf16=True` — a T4 (common on free-tier Colab) doesn't support bf16 and would raise an error on it, while an A100/L4 does.
- `free_memory()` clears GPU cache and runs garbage collection between models, so the second and third model don't fight the first one for GPU memory.



In [18]:
def get_precision_kwargs() -> dict:
    """
    Picks the fastest safe mixed-precision setting for the current hardware.
    Tesla T4 (free Colab) does NOT support bf16.
    """
    if not torch.cuda.is_available():
        return {'fp16': False, 'bf16': False}

    # Explicit check for compute capability: bf16 requires >= 8.0 (Ampere+)
    # T4 is 7.5 (Turing), so it must use fp16.
    if torch.cuda.get_device_capability()[0] >= 8:
        try:
            if torch.cuda.is_bf16_supported():
                return {'fp16': False, 'bf16': True}
        except Exception:
            pass

    return {'fp16': True, 'bf16': False}

print("Precision setting for this runtime:", get_precision_kwargs())

Precision setting for this runtime: {'fp16': False, 'bf16': True}


In [19]:
def get_optimizer_params(model, learning_rate, weight_decay, layer_decay=0.9):
    """
    Groups model parameters with layer-wise learning rate decay.
    Lower layers get a smaller learning rate.
    """
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    optimizer_grouped_parameters = []

    # 1. Classification head (Maximum LR)
    head_params = [p for n, p in model.named_parameters() if 'classifier' in n or 'pooler' in n]
    if head_params:
        optimizer_grouped_parameters.append({
            'params': head_params,
            'lr': learning_rate,
            'weight_decay': weight_decay
        })

    # 2. Transformer Layers (Decaying LR)
    # Most architectures (BERT, RoBERTa, MuRIL) use 'layer.i' in their naming convention
    for layer_idx in range(11, -1, -1):
        layer_lr = learning_rate * (layer_decay ** (11 - layer_idx))

        optimizer_grouped_parameters.extend([
            {
                'params': [p for n, p in model.named_parameters() if f'layer.{layer_idx}.' in n and not any(nd in n for nd in no_decay)],
                'lr': layer_lr,
                'weight_decay': weight_decay
            },
            {
                'params': [p for n, p in model.named_parameters() if f'layer.{layer_idx}.' in n and any(nd in n for nd in no_decay)],
                'lr': layer_lr,
                'weight_decay': 0.0
            }
        ])

    # 3. Embeddings (Lowest LR)
    embed_params = [p for n, p in model.named_parameters() if 'embeddings' in n]
    if embed_params:
        optimizer_grouped_parameters.append({
            'params': embed_params,
            'lr': learning_rate * (layer_decay ** 12),
            'weight_decay': weight_decay
        })

    return optimizer_grouped_parameters

## 10. Unified Training Pipeline

This is the piece that was missing for mBERT and MuRIL: **one function**, used for every model, that tokenizes, builds datasets, initializes the model, trains with `WeightedTrainer`, evaluates, plots a confusion matrix, saves to Drive, and cleans up — always in that order, so evaluation and saving always happen *before* anything gets deleted.

In [20]:
"""
RESEARCH-GRADE SENTIMENT PIPELINE — v6.1 (SPEED-OPTIMIZED, SINGLE SEED)
=========================================================================

Changes from v6:
  [SPEED]  Single seed only (no multi-seed loop)
  [SPEED]  gradient_checkpointing OFF by default (configurable)
  [SPEED]  dataloader_num_workers=0 (avoids Windows/Jupyter deadlock)
  [SPEED]  Train diagnostics on SUBSET (default 1000 samples)
  [SPEED]  save_only_model=True (smaller checkpoints, faster I/O)
  [SPEED]  save_total_limit=1 (less disk writes)
  [SPEED]  batch_size=32, grad_accum=1 (configurable)
  [SPEED]  Sampled token-length analysis (5000 samples)
  [SPEED]  tf32=True + auto bf16/fp16
  [SPEED]  logging_strategy="steps" for live progress visibility
  [KEEP]   Dynamic padding via DataCollatorWithPadding
  [KEEP]   Early stopping, best-checkpoint loading
  [KEEP]   Bootstrap CI, ECE, ROC, error analysis (on TEST only)

Assumes the following exist earlier in your notebook/module:
    Constants:
        MAX_LEN, SEED, EPOCHS, EARLY_STOP_PATIENCE,
        GRAD_ACCUM, BATCH_TRAIN, BATCH_EVAL,
        LR, WEIGHT_DECAY, N_LABELS, ID2LABEL, SENT_MAP
    Classes:
        SentimentDataset, CustomSentimentModel, WeightedTrainer
    Functions:
        get_optimizer_params(model, lr, weight_decay, layer_decay=...),
        get_precision_kwargs(), compute_metrics(eval_pred)
"""

import os
import gc
import sys
import json
import shutil
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import optuna
import transformers
import sklearn
import math
import matplotlib
matplotlib.use("Agg")  # headless-safe backend
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    DataCollatorWithPadding,
)

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report,
    f1_score, accuracy_score, matthews_corrcoef,
    average_precision_score, roc_auc_score, balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize


# ================================================================
# 0. REPRODUCIBILITY & ENVIRONMENT
# ================================================================

def set_seed(seed):
    """Set ALL random seeds for full reproducibility (single seed)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic algorithms for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_environment_info():
    return {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else "N/A",
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
        "precision_mode": "bf16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "fp16",
    }


def _clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _log(msg=""):
    """Print with flush to avoid silent-hang appearance in notebooks."""
    print(msg, flush=True)


# ================================================================
# 1. DATA CLEANING / MAPPING / STRATIFIED SPLIT
# ================================================================

def prepare_dataset(df, text_col="input_text", label_col="sentiment_raw",
                    seed=42, test_size=0.1):
    _log("\n" + "=" * 80)
    _log("  DATA PREPARATION")
    _log("=" * 80)

    initial = len(df)
    df = df.dropna(subset=[text_col]).drop_duplicates(subset=[text_col]).reset_index(drop=True)
    _log(f"  Rows after cleaning: {len(df)} (removed {initial - len(df)})")

    if "label" not in df.columns or df["label"].dtype == object:
        df["label"] = df[label_col].map(SENT_MAP)
        df = df.dropna(subset=["label"]).reset_index(drop=True)
        df["label"] = df["label"].astype(int)

    train_val_df, test_df = train_test_split(
        df, test_size=test_size, stratify=df["label"], random_state=seed,
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=(test_size / (1.0 - test_size)),
        stratify=train_val_df["label"], random_state=seed,
    )

    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    _log(f"\n  Train : {len(train_df)} rows")
    _log(f"  Val   : {len(val_df)} rows")
    _log(f"  Test  : {len(test_df)} rows")
    _log(f"\n  Train dist: {train_df['label'].value_counts().to_dict()}")
    _log(f"  Val   dist: {val_df['label'].value_counts().to_dict()}")
    _log(f"  Test  dist: {test_df['label'].value_counts().to_dict()}")

    classes = np.unique(train_df["label"].to_numpy())
    cw = compute_class_weight(class_weight="balanced", classes=classes,
                              y=train_df["label"].to_numpy())
    class_weights = torch.tensor(cw, dtype=torch.float)
    _log(f"\n  ⚖️ Class weights: {class_weights.numpy().round(4)}")

    return train_df, val_df, test_df, class_weights


# ================================================================
# 2. STRICT DATA VALIDATION
# ================================================================

def validate_data(train_df, val_df, test_df, text_col="input_text", strict=True):
    report = {}
    report["missing"] = {
        s: int(df[text_col].isna().sum())
        for s, df in zip(["train", "val", "test"], [train_df, val_df, test_df])
    }
    report["duplicates"] = {}
    report["duplicate_label_conflicts"] = {}
    for s, df in zip(["train", "val", "test"], [train_df, val_df, test_df]):
        dup_mask = df.duplicated(subset=[text_col], keep=False)
        dup_df = df[dup_mask].copy()
        report["duplicates"][s] = int(len(dup_df))
        if dup_df.empty:
            report["duplicate_label_conflicts"][s] = 0
            continue
        conflicting = dup_df.groupby(text_col)["label"].nunique()
        report["duplicate_label_conflicts"][s] = int((conflicting > 1).sum())

    report["label_conflicts"] = {}
    for s, df in zip(["train", "val", "test"], [train_df, val_df, test_df]):
        lc = df.groupby(text_col)["label"].nunique()
        report["label_conflicts"][s] = int((lc > 1).sum())

    sets = {s: set(df[text_col].astype(str))
            for s, df in zip(["train", "val", "test"], [train_df, val_df, test_df])}
    report["leakage"] = {
        "train_val": len(sets["train"] & sets["val"]),
        "train_test": len(sets["train"] & sets["test"]),
        "val_test": len(sets["val"] & sets["test"]),
    }
    report["class_distribution"] = {
        s: df["label"].value_counts().to_dict()
        for s, df in zip(["train", "val", "test"], [train_df, val_df, test_df])
    }

    if strict:
        if any(v > 0 for v in report["missing"].values()):
            raise ValueError(f"❌ Missing values: {report['missing']}")
        if any(v > 0 for v in report["duplicate_label_conflicts"].values()):
            raise ValueError(f"❌ Duplicate label conflicts: {report['duplicate_label_conflicts']}")
        if any(v > 0 for v in report["leakage"].values()):
            raise ValueError(f"❌ Cross-split leakage: {report['leakage']}")
        if any(v > 0 for v in report["label_conflicts"].values()):
            raise ValueError(f"❌ Label conflicts: {report['label_conflicts']}")
    return report


# ================================================================
# 3. TOKEN LENGTH ANALYSIS (SAMPLED FOR SPEED)
# ================================================================

def analyze_token_lengths(texts, tokenizer, current_max_len, sample_size=5000):
    texts = list(texts)
    sampled = False
    if sample_size and len(texts) > sample_size:
        idx = np.random.RandomState(42).choice(len(texts), sample_size, replace=False)
        texts = [texts[i] for i in idx]
        sampled = True

    lengths = [len(tokenizer.encode(str(x), add_special_tokens=True)) for x in texts]
    truncation_rate = float(np.mean(np.array(lengths) > current_max_len) * 100)

    _log("\n  [Token Length Distribution]" + (f"  (sampled {len(texts)})" if sampled else ""))
    for p in [90, 95, 97, 98, 99, 99.5, 100]:
        _log(f"    {p:>5.1f}% → {np.percentile(lengths, p):.0f} tokens")
    _log(f"    Current MAX_LEN = {current_max_len}")
    _log(f"    Truncation Rate = {truncation_rate:.2f}%")
    return lengths, truncation_rate


# ================================================================
# 4. PAIRED BOOTSTRAP CI
# ================================================================

def compute_bootstrap(y_true, y_pred, target_names, n_bootstrap=1000, ci=0.95, seed=42):
    n = len(y_true)
    rng = np.random.RandomState(seed)
    acc_s, mac_s, wei_s, mcc_s, bal_s = [], [], [], [], []
    pc_f1_s = {cls: [] for cls in target_names}

    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        acc_s.append(accuracy_score(yt, yp))
        mac_s.append(f1_score(yt, yp, average="macro", zero_division=0))
        wei_s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        mcc_s.append(matthews_corrcoef(yt, yp))
        bal_s.append(balanced_accuracy_score(yt, yp))
        pc = f1_score(yt, yp, average=None, labels=list(range(len(target_names))), zero_division=0)
        for i, cls in enumerate(target_names):
            pc_f1_s[cls].append(pc[i])

    def get_ci(scores):
        s = np.array(scores)
        return (float(np.mean(s)),
                float(np.percentile(s, (1 - ci) / 2 * 100)),
                float(np.percentile(s, (1 + ci) / 2 * 100)))

    return {
        "accuracy": get_ci(acc_s), "macro_f1": get_ci(mac_s),
        "weighted_f1": get_ci(wei_s), "mcc": get_ci(mcc_s),
        "balanced_accuracy": get_ci(bal_s),
        "per_class_f1": {cls: get_ci(scores) for cls, scores in pc_f1_s.items()},
    }


# ================================================================
# 5. CALIBRATION & ECE
# ================================================================

def compute_ece_and_plot(y_true, y_probs, target_names, save_path, n_bins=10):
    fig, axes = plt.subplots(1, len(target_names), figsize=(5 * len(target_names), 5))
    if len(target_names) == 1:
        axes = [axes]
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece_scores = {}

    for i, cls in enumerate(target_names):
        y_bin = (y_true == i).astype(int)
        prob_pos = y_probs[:, i]
        if y_bin.sum() == 0:
            axes[i].text(0.5, 0.5, "Not in test set", ha="center", va="center")
            ece_scores[cls] = None
            continue
        ece, bin_accs, bin_confs = 0.0, [], []
        for b in range(n_bins):
            lo, hi = bin_edges[b], bin_edges[b + 1]
            mask = (prob_pos >= lo) & (prob_pos <= hi if b == n_bins - 1 else prob_pos < hi)
            count = mask.sum()
            if count > 0:
                acc, conf = float(y_bin[mask].mean()), float(prob_pos[mask].mean())
                ece += (count / len(y_bin)) * abs(acc - conf)
                bin_accs.append(acc)
                bin_confs.append(conf)
            else:
                bin_accs.append(np.nan)
                bin_confs.append(np.nan)
        ece_scores[cls] = float(ece)
        valid = ~np.isnan(bin_accs)
        axes[i].plot(np.array(bin_confs)[valid], np.array(bin_accs)[valid], "s-", label=cls)
        axes[i].plot([0, 1], [0, 1], "k--", label="Perfect")
        axes[i].set_title(f"{cls}\nECE = {ece:.4f}")
        axes[i].set_xlabel("Confidence")
        axes[i].set_ylabel("Accuracy")
        axes[i].set_xlim(0, 1)
        axes[i].set_ylim(0, 1)
        axes[i].grid(True, alpha=0.5)

    plt.suptitle("Probability Calibration (TEST SET)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

    valid_ece = [v for v in ece_scores.values() if v is not None]
    mean_class_ece = float(np.mean(valid_ece)) if valid_ece else None
    return ece_scores, mean_class_ece


# ================================================================
# 6. LABEL PREPARATION
# ================================================================

def _prepare_labels(label_series: pd.Series) -> list:
    if pd.api.types.is_numeric_dtype(label_series):
        return label_series.astype(int).tolist()
    try:
        return label_series.astype(int).tolist()
    except Exception:
        return label_series.map(SENT_MAP).astype(int).tolist()


# ================================================================
# 7. EPOCH METRICS PRINTER CALLBACK
# ================================================================

class EpochMetricsPrinter(TrainerCallback):
    """Prints epoch-level metrics in old-style format."""

    def __init__(self):
        self.last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs and "eval_loss" not in logs:
            self.last_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None or state.epoch is None:
            return
        epoch = int(round(state.epoch))
        total_epochs = int(args.num_train_epochs)
        _log("\n" + "=" * 62)
        _log(f"  📊 EPOCH {epoch}/{total_epochs}")
        _log("=" * 62)
        if self.last_train_loss is not None:
            _log(f"  Train Loss          : {self.last_train_loss:.6f}")
        else:
            _log("  Train Loss          : N/A")
        val_loss = metrics.get("eval_loss")
        val_acc  = metrics.get("eval_accuracy")
        val_f1   = metrics.get("eval_macro_f1")
        if val_loss is not None:
            _log(f"  Validation Loss     : {val_loss:.6f}")
        if val_acc is not None:
            _log(f"  Validation Accuracy : {val_acc:.6f}")
        if val_f1 is not None:
            _log(f"  Validation Macro-F1 : {val_f1:.6f}")
        _log("=" * 62 + "\n")


# ================================================================
# 8. LLRD + WEIGHTED TRAINER
# ================================================================

class LLRDWeightedTrainer(WeightedTrainer):
    def __init__(self, *args, use_llrd=False, layer_decay=0.9, **kwargs):
        self.use_llrd = use_llrd
        self.layer_decay = layer_decay
        super().__init__(*args, **kwargs)

    def _reset_optimizer_and_scheduler(self):
        self.optimizer = None
        self.lr_scheduler = None

    def _hp_search_setup(self, trial):
        self._reset_optimizer_and_scheduler()
        try:
            super()._hp_search_setup(trial)
        except AttributeError:
            params = self.hp_space(trial)
            for key, value in params.items():
                setattr(self.args, key, value)

    def train(self, *args, **kwargs):
        trial = kwargs.get("trial", None)
        if trial is None and len(args) >= 2:
            trial = args[1]
        if trial is not None:
            self._reset_optimizer_and_scheduler()
        return super().train(*args, **kwargs)

    def create_optimizer(self):
        if getattr(self, "optimizer", None) is not None:
            return self.optimizer
        if not self.use_llrd:
            optimizer = super().create_optimizer()
            if getattr(self, "optimizer", None) is None:
                self.optimizer = optimizer
            return self.optimizer
        grouped_params = get_optimizer_params(
            self.model, self.args.learning_rate,
            self.args.weight_decay, layer_decay=self.layer_decay,
        )
        self.optimizer = torch.optim.AdamW(
            grouped_params,
            lr=self.args.learning_rate,
            weight_decay=self.args.weight_decay,
        )
        return self.optimizer


# ================================================================
# 9. MAIN PIPELINE v6.1 (SPEED-OPTIMIZED, SINGLE SEED)
# ================================================================

def run_pipeline_v6(config: dict, train_df: pd.DataFrame, val_df: pd.DataFrame,
                    test_df: pd.DataFrame, class_weights: torch.Tensor,
                    report_to: str = "none", n_trials: int = 5) -> dict:

    # 🔴 SINGLE SEED — set once, used everywhere
    set_seed(SEED)

    # ── READ CONFIG ──
    key        = config["key"]
    label      = config["label"]
    checkpoint = config["checkpoint"]
    save_dir   = config["save_dir"]
    output_dir = config["output_dir"]
    tok_dir    = config["tok_dir"]

    use_llrd         = config.get("use_llrd", False)
    layer_decay      = config.get("layer_decay", 0.9)
    use_optuna       = config.get("use_optuna", False)
    use_custom_model = config.get("use_custom_model", False)
    search_epochs    = config.get("search_epochs", 4)
    default_warmup   = config.get("warmup_ratio", 0.1)
    batch_candidates = config.get("batch_size_candidates", [16, 32])
    wd_min, wd_max   = config.get("weight_decay_range", (0.01, 0.20))
    save_total_limit = config.get("save_total_limit", 1)  # 🔴 SPEED: reduced from 2

    # 🔴 SPEED CONFIG
    grad_checkpointing  = config.get("gradient_checkpointing", False)  # OFF by default
    train_diag_size     = config.get("train_diagnostic_size", 1000)    # subset for diagnostics
    num_workers         = config.get("num_workers", 0)                 # avoid deadlock
    batch_train         = config.get("batch_size", BATCH_TRAIN)
    grad_accum          = config.get("grad_accum", GRAD_ACCUM)

    # ── DIRECTORIES ──
    plots_dir = os.path.join(save_dir, "plots")
    model_dir = os.path.join(save_dir, "model")
    for d in [save_dir, tok_dir, output_dir, plots_dir, model_dir]:
        os.makedirs(d, exist_ok=True)

    if os.path.abspath(save_dir) == os.path.abspath(output_dir):
        raise ValueError("save_dir and output_dir must be different.")

    _log("\n" + "=" * 80)
    _log(f"  PIPELINE v6.1 (SPEED-OPTIMIZED): {label}")
    _log("=" * 80)
    _log(f"  Seed            : {SEED} (single)")
    _log(f"  Optuna          : {use_optuna}")
    _log(f"  LLRD            : {use_llrd}")
    _log(f"  Grad Ckpt       : {grad_checkpointing}")
    _log(f"  Batch/Accum     : {batch_train}/{grad_accum}")
    _log(f"  Learning Rate   : {config.get('learning_rate', LR)}")

    # ============================================================
    # STEP 1: STRICT QA
    # ============================================================
    _log("\n[1] Strict Data QA")
    data_report = validate_data(train_df, val_df, test_df, strict=True)
    _log("  ✅ Passed strict QA")

    # ============================================================
    # STEP 2: TOKENIZATION
    # ============================================================
    _log("\n[2] Tokenization & Length Analysis")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    _, truncation_rate = analyze_token_lengths(
        train_df["input_text"], tokenizer, MAX_LEN, sample_size=5000,
    )

    # 🔴 Dynamic padding: padding=False here, DataCollator pads per-batch
    train_enc = tokenizer(train_df["input_text"].astype(str).tolist(),
                          truncation=True, padding=False, max_length=MAX_LEN)
    val_enc   = tokenizer(val_df["input_text"].astype(str).tolist(),
                          truncation=True, padding=False, max_length=MAX_LEN)
    test_enc  = tokenizer(test_df["input_text"].astype(str).tolist(),
                          truncation=True, padding=False, max_length=MAX_LEN)

    train_labels = _prepare_labels(train_df["label"])
    val_labels   = _prepare_labels(val_df["label"])
    test_labels  = _prepare_labels(test_df["label"])

    train_ds = SentimentDataset(train_enc, train_labels)
    val_ds   = SentimentDataset(val_enc, val_labels)
    test_ds  = SentimentDataset(test_enc, test_labels)

    labels       = list(range(N_LABELS))
    target_names = [str(ID2LABEL[i]) for i in sorted(ID2LABEL.keys())]
    _log(f"  Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

    # ── MODEL INIT ──
    def model_init(trial=None):
        if use_custom_model:
            return CustomSentimentModel(checkpoint, N_LABELS, ID2LABEL, SENT_MAP)
        return AutoModelForSequenceClassification.from_pretrained(
            checkpoint, num_labels=N_LABELS, id2label=ID2LABEL, label2id=SENT_MAP,
        )

    # ── Dynamic padding collator ──
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer, padding=True, pad_to_multiple_of=8,
    )

    # ============================================================
    # STEP 3: OPTUNA PHASE 1 (optional)
    # ============================================================
    best_hparams = {}
    best_objective = None

    if use_optuna:
        _log("\n" + "-" * 80)
        _log("  PHASE 1: OPTUNA SEARCH (Train + Val only)")
        _log("-" * 80)

        search_args = TrainingArguments(
            output_dir=output_dir,
            optim="adamw_torch",
            num_train_epochs=search_epochs,
            per_device_train_batch_size=batch_train,
            per_device_eval_batch_size=BATCH_EVAL,
            learning_rate=config.get("learning_rate", LR),
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=default_warmup,
            gradient_accumulation_steps=grad_accum,
            seed=SEED,
            eval_strategy="epoch",
            save_strategy="no",              # 🔴 SPEED: no checkpoints during search
            load_best_model_at_end=False,
            gradient_checkpointing=grad_checkpointing,
            logging_strategy="epoch",
            logging_steps=10,
            disable_tqdm=True,
            report_to="none",
            tf32=torch.cuda.is_available(),
            dataloader_pin_memory=True,
            dataloader_num_workers=num_workers,
            **get_precision_kwargs(),
        )

        search_trainer = LLRDWeightedTrainer(
            model_init=model_init, args=search_args,
            train_dataset=train_ds, eval_dataset=val_ds,
            compute_metrics=compute_metrics, class_weights=class_weights,
            data_collator=data_collator, use_llrd=use_llrd, layer_decay=layer_decay,
            callbacks=[EpochMetricsPrinter()],
        )

        def hp_space(trial):
            return {
                "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
                "weight_decay": trial.suggest_float("weight_decay", wd_min, wd_max),
                "warmup_ratio": trial.suggest_float("warmup_ratio", 0.05, 0.20),
                "per_device_train_batch_size": trial.suggest_categorical(
                    "per_device_train_batch_size", batch_candidates),
            }

        try:
            best_run = search_trainer.hyperparameter_search(
                direction="maximize", backend="optuna", hp_space=hp_space,
                n_trials=n_trials, compute_objective=lambda m: m["eval_macro_f1"],
                sampler=optuna.samplers.TPESampler(seed=SEED),
            )
        except TypeError:
            best_run = search_trainer.hyperparameter_search(
                direction="maximize", backend="optuna", hp_space=hp_space,
                n_trials=n_trials, compute_objective=lambda m: m["eval_macro_f1"],
            )

        best_hparams   = dict(best_run.hyperparameters)
        best_objective = best_run.objective
        _log(f"\n  BEST OPTUNA Macro-F1: {best_objective:.4f}")
        for k, v in best_hparams.items():
            _log(f"    {k:<35}: {v}")

        del search_trainer
        _clear_memory()

    else:
        _log("\n  Optuna disabled → Using fixed hyperparameters.")
        best_hparams = {
            "learning_rate": config.get("learning_rate", LR),
            "weight_decay": config.get("weight_decay", WEIGHT_DECAY),
            "warmup_ratio": config.get("warmup_ratio", default_warmup),
            "per_device_train_batch_size": config.get("batch_size", BATCH_TRAIN),
        }
        for k, v in best_hparams.items():
            _log(f"    {k:<35}: {v}")

    # ============================================================
    # STEP 4: FINAL TRAINING (SINGLE SEED)
    # ============================================================
    _log("\n" + "-" * 80)
    _log("  PHASE 2: FINAL TRAINING (single seed)")
    _log("-" * 80)

    final_args = TrainingArguments(
        output_dir=output_dir,
        optim="adamw_torch",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=best_hparams.get("per_device_train_batch_size", batch_train),
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=best_hparams.get("learning_rate", LR),
        weight_decay=best_hparams.get("weight_decay", WEIGHT_DECAY),
        warmup_ratio=best_hparams.get("warmup_ratio", default_warmup),
        gradient_accumulation_steps=grad_accum,
        seed=SEED,                          # 🔴 SINGLE SEED
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=save_total_limit,  # 🔴 SPEED: 1 checkpoint max
        save_only_model=True,               # 🔴 SPEED: skip optimizer/scheduler state
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_strategy="steps",           # 🔴 SPEED: visible progress
        logging_steps=25,
        logging_first_step=True,
        disable_tqdm=False,
        gradient_checkpointing=grad_checkpointing,  # 🔴 SPEED: OFF by default
        report_to=report_to,
        tf32=torch.cuda.is_available(),
        dataloader_pin_memory=True,
        dataloader_num_workers=num_workers,  # 🔴 SPEED: 0 avoids deadlock
        **get_precision_kwargs(),
    )

    final_trainer = LLRDWeightedTrainer(
        model_init=model_init, args=final_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics, class_weights=class_weights,
        data_collator=data_collator, use_llrd=use_llrd, layer_decay=layer_decay,
        callbacks=[
            EpochMetricsPrinter(),
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
        ],
    )

    final_trainer.train()
    _log("\n  ✅ Training complete. Best checkpoint loaded.")

    # ── BEST CHECKPOINT INFO ──
    best_epoch      = getattr(final_trainer.state, "best_epoch", None)
    best_metric     = getattr(final_trainer.state, "best_metric", None)
    best_checkpoint = getattr(final_trainer.state, "best_model_checkpoint", None)
    global_step     = getattr(final_trainer.state, "global_step", None)

    # ============================================================
    # STEP 5: TRAIN vs VAL DIAGNOSTICS (SUBSET FOR SPEED)
    # ============================================================
    _log("\n[Train vs Validation Diagnostics]")

    # 🔴 SPEED: Evaluate only a SUBSET of training data
    train_diag_size = min(train_diag_size, len(train_ds))
    train_diag_ds = torch.utils.data.Subset(train_ds, list(range(train_diag_size)))

    tr_m = final_trainer.evaluate(train_diag_ds, metric_key_prefix="train")
    vl_m = final_trainer.evaluate(val_ds, metric_key_prefix="val")

    tr_f1 = tr_m.get("train_macro_f1", 0)
    vl_f1 = vl_m.get("val_macro_f1", 0)
    train_val_gap = tr_f1 - vl_f1

    if train_val_gap > 0.05:
        gap_status = "⚠️ Potential overfitting (Gap > 0.05)"
    elif train_val_gap < -0.01:
        gap_status = "⚠️ Potential underfitting (Gap < -0.01)"
    else:
        gap_status = "✅ Healthy generalization"

    _log(f"  Train Macro-F1 ({train_diag_size} samples): {tr_f1:.4f}")
    _log(f"  Val   Macro-F1 : {vl_f1:.4f}")
    _log(f"  Gap            : {train_val_gap:+.4f}")
    _log(f"  Status         : {gap_status}")

    # ============================================================
    # STEP 6: INDEPENDENT TEST EVALUATION
    # ============================================================
    _log("\n" + "-" * 80)
    _log("  INDEPENDENT TEST EVALUATION")
    _log("-" * 80)

    test_pred = final_trainer.predict(test_ds)
    logits = test_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits, dtype=np.float32)
    y_true = np.asarray(test_pred.label_ids, dtype=np.int64)

    if N_LABELS == 2 and logits.ndim == 2 and logits.shape[1] == 1:
        prob_pos = torch.sigmoid(torch.tensor(logits[:, 0])).numpy()
        y_probs = np.column_stack([1 - prob_pos, prob_pos])
        y_pred  = (prob_pos >= 0.5).astype(int)
    elif N_LABELS == 2 and logits.ndim == 1:
        prob_pos = torch.sigmoid(torch.tensor(logits)).numpy()
        y_probs = np.column_stack([1 - prob_pos, prob_pos])
        y_pred  = (prob_pos >= 0.5).astype(int)
    else:
        y_pred  = np.argmax(logits, axis=-1)
        y_probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()

    # ── METRICS ──
    report = classification_report(
        y_true, y_pred, labels=labels, target_names=target_names,
        output_dict=True, zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    per_class_f1 = {
        cls: float(report[cls]["f1-score"]) if cls in report else 0.0
        for cls in target_names
    }

    mcc          = matthews_corrcoef(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    test_f1      = f1_score(y_true, y_pred, average="macro", zero_division=0)
    val_test_gap = vl_f1 - test_f1

    y_bin = label_binarize(y_true, classes=labels)
    roc_auc, pr_auc = {}, {}
    for i in range(N_LABELS):
        if len(np.unique(y_bin[:, i])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        roc_auc[target_names[i]] = float(auc(fpr, tpr))
        pr_auc[target_names[i]]  = float(average_precision_score(y_bin[:, i], y_probs[:, i]))

    try:
        macro_roc_auc = roc_auc_score(y_true, y_probs, multi_class="ovr", average="macro")
    except ValueError:
        macro_roc_auc = None

    _log(f"\n  Accuracy          : {report['accuracy']:.4f}")
    _log(f"  Balanced Accuracy : {balanced_acc:.4f}")
    _log(f"  Macro-F1          : {test_f1:.4f}")
    _log(f"  Weighted-F1       : {report['weighted avg']['f1-score']:.4f}")
    _log(f"  MCC               : {mcc:.4f}")
    if macro_roc_auc is not None:
        _log(f"  Macro ROC-AUC     : {macro_roc_auc:.4f}")
    _log(f"  Val→Test Gap      : {val_test_gap:+.4f}")

    # ── BOOTSTRAP CI ──
    _log("\n[Bootstrap CI (n=1000)]")
    boot = compute_bootstrap(y_true, y_pred, target_names)
    for metric, (mean, lo, hi) in boot.items():
        if metric != "per_class_f1":
            _log(f"  {metric:<20}: {mean:.4f} [{lo:.4f}, {hi:.4f}]")

    # ── CALIBRATION ──
    _log("\n[Calibration / ECE]")
    ece_scores, mean_class_ece = compute_ece_and_plot(
        y_true, y_probs, target_names,
        save_path=os.path.join(plots_dir, "calibration_test.png"),
    )
    for cls, ece in ece_scores.items():
        if ece is not None:
            s = "✅" if ece < 0.05 else ("🟠" if ece < 0.10 else "⚠️")
            _log(f"  {cls:<15}: ECE = {ece:.4f} {s}")
    if mean_class_ece is not None:
        _log(f"  Mean Class ECE  : {mean_class_ece:.4f}")

    # ── GENERALIZATION SUMMARY ──
    _log("\n" + "=" * 80)
    _log("  GENERALIZATION ANALYSIS")
    _log("=" * 80)
    _log(f"  Train F1  : {tr_f1:.4f}")
    _log(f"  Val   F1  : {vl_f1:.4f}")
    _log(f"  Test  F1  : {test_f1:.4f}")
    _log(f"  Train→Val : {train_val_gap:+.4f}")
    _log(f"  Val→Test  : {val_test_gap:+.4f}")
    _log(f"  Status    : {gap_status}")

    # ── ERROR ANALYSIS ──
    _log("\n[Error Analysis]")
    lengths = [
        len(tokenizer.encode(str(test_df["input_text"].iloc[i]), add_special_tokens=True))
        for i in range(len(y_true))
    ]
    errors = []
    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            errors.append({
                "text": test_df["input_text"].iloc[i],
                "true": target_names[y_true[i]],
                "pred": target_names[y_pred[i]],
                "confidence": float(y_probs[i, y_pred[i]]),
                "token_length": lengths[i],
            })
    errors_df = pd.DataFrame(errors).sort_values("confidence", ascending=False)
    errors_df.to_csv(os.path.join(save_dir, "error_analysis.csv"), index=False)
    _log(f"  Saved {len(errors_df)} errors → error_analysis.csv")

    # ── VISUALIZATIONS ──
    _log("\n[Saving Plots]")
    history = final_trainer.state.log_history
    hist_df = pd.DataFrame(history)

    # Loss curve
    tl = [l["loss"] for l in history if "loss" in l and "eval_loss" not in l]
    te = [l["epoch"] for l in history if "loss" in l and "eval_loss" not in l]
    vl_loss = [l["eval_loss"] for l in history if "eval_loss" in l]
    ve = [l["epoch"] for l in history if "eval_loss" in l]

    plt.figure(figsize=(10, 5))
    if tl:
        plt.plot(te, tl, marker="o", label="Train Loss")
    if vl_loss:
        plt.plot(ve, vl_loss, marker="o", label="Validation Loss")
    plt.title(f"Loss Curves: {label}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "loss_curve.png"), dpi=150)
    plt.close()

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f"Confusion Matrix (TEST): {label}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    # ROC curves
    plt.figure(figsize=(8, 6))
    for i in range(N_LABELS):
        if len(np.unique(y_bin[:, i])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        plt.plot(fpr, tpr, lw=2, label=f"{target_names[i]} (AUC={auc(fpr, tpr):.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"ROC Curves (TEST): {label}")
    plt.legend(loc="lower right")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "roc_test.png"), dpi=150)
    plt.close()

    # ============================================================
    # SAVE ARTIFACTS
    # ============================================================
    _log("\n[Saving Artifacts]")

    if hasattr(final_trainer.model, "backbone") and \
       hasattr(final_trainer.model.backbone, "gradient_checkpointing_disable"):
        final_trainer.model.backbone.gradient_checkpointing_disable()

    for p in final_trainer.model.parameters():
        p.data = p.data.contiguous()

    final_trainer.save_model(model_dir)
    tokenizer.save_pretrained(tok_dir)

    with open(os.path.join(save_dir, "config.json"), "w") as f:
        json.dump(config, f, indent=2, default=str)

    metrics_out = {
        "test_n": len(y_true),
        "class_distributions": data_report["class_distribution"],
        "accuracy": float(report["accuracy"]),
        "balanced_accuracy": float(balanced_acc),
        "macro_f1": float(test_f1),
        "weighted_f1": float(report["weighted avg"]["f1-score"]),
        "mcc": float(mcc),
        "macro_roc_auc": float(macro_roc_auc) if macro_roc_auc is not None else None,
        "mean_class_ece": mean_class_ece,
        "truncation_rate": float(truncation_rate),
        "per_class_f1": per_class_f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "train_f1": float(tr_f1),
        "val_f1": float(vl_f1),
        "test_f1": float(test_f1),
        "train_val_gap": float(train_val_gap),
        "val_test_gap": float(val_test_gap),
        "gap_status": gap_status,
        "best_epoch": best_epoch,
        "best_val_metric": best_metric,
        "best_checkpoint": str(best_checkpoint) if best_checkpoint else None,
        "global_step": global_step,
        "best_hparams": best_hparams,
        "bootstrap_ci": boot,
        "ece_scores": ece_scores,
        "data_validation": {k: v for k, v in data_report.items() if k != "class_distribution"},
        "environment": get_environment_info(),
    }
    with open(os.path.join(save_dir, "metrics.json"), "w") as f:
        json.dump(metrics_out, f, indent=2, default=str)

    report_df.to_csv(os.path.join(save_dir, "classification_report.csv"))
    hist_df.to_csv(os.path.join(save_dir, "training_history.csv"), index=False)

    preds_df = pd.DataFrame({"true_label": y_true, "predicted_label": y_pred})
    for i, cls in enumerate(target_names):
        preds_df[f"prob_{cls}"] = y_probs[:, i]
    preds_df.to_csv(os.path.join(save_dir, "test_predictions.csv"), index=False)

    if best_hparams:
        pd.DataFrame({
            "parameter": list(best_hparams.keys()),
            "value": list(best_hparams.values()),
        }).to_csv(os.path.join(save_dir, "best_hyperparameters.csv"), index=False)

    _log(f"  Model     → {model_dir}")
    _log(f"  Tokenizer → {tok_dir}")
    _log(f"  Plots     → {plots_dir}")

    del final_trainer
    _clear_memory()

    _log("\n" + "=" * 80)
    _log("  PIPELINE v6.1 COMPLETED SUCCESSFULLY")
    _log("=" * 80)

    return {
        "label": label,
        "best_hparams": best_hparams,
        "test_accuracy": float(report["accuracy"]),
        "test_macro_f1": float(test_f1),
        "test_weighted_f1": float(report["weighted avg"]["f1-score"]),
        "balanced_accuracy": float(balanced_acc),
        "mcc": float(mcc),
        "macro_roc_auc": macro_roc_auc,
        "mean_class_ece": mean_class_ece,
        "per_class_f1": per_class_f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "train_f1": float(tr_f1),
        "val_f1": float(vl_f1),
        "train_val_gap": float(train_val_gap),
        "val_test_gap": float(val_test_gap),
        "gap_status": gap_status,
        "bootstrap_ci": boot,
        "ece_scores": ece_scores,
        "classification_report": report_df,
        "history": hist_df,
        "predictions_df": preds_df,
        "errors_df": errors_df,
    }


# ================================================================
# 10. RUNNER
# ================================================================

if "run_history" not in globals():
    run_history = []
if "all_results" not in globals():
    all_results = {}

BASE_CONFIGS = {
    "muril": {"key": "muril", "label": "MuRIL", "checkpoint": "google/muril-base-cased"},
    "xlmr":  {"key": "xlmr",  "label": "XLM-RoBERTa", "checkpoint": "xlm-roberta-base"},
    "mbert": {"key": "mbert", "label": "mBERT", "checkpoint": "bert-base-multilingual-cased"},
}

DEFAULT_HPARAMS = {
    "learning_rate": 1e-5,
    "weight_decay": 0.15,
    "warmup_ratio": 0.1,
    "batch_size": 32,                     # 🔴 SPEED: larger batch
    "grad_accum": 1,                     # 🔴 SPEED: no accumulation needed
    "use_llrd": False,
    "layer_decay": 0,
    "use_custom_model": False,
    "use_optuna": False,
    "search_epochs": 5,
    "batch_size_candidates": [16, 32],
    "weight_decay_range": (0.01, 0.20),
    "save_total_limit": 1,               # 🔴 SPEED
    "gradient_checkpointing": False,     # 🔴 SPEED: OFF by default
    "train_diagnostic_size": 1000,       # 🔴 SPEED: subset for diagnostics
    "num_workers": 0,                    # 🔴 SPEED: avoids deadlock
}


def run_model(model_key, run_name=None, clean_old=True, n_trials=5, **custom_hparams):
    if model_key not in BASE_CONFIGS:
        raise ValueError(f"Unknown '{model_key}'. Available: {list(BASE_CONFIGS.keys())}")

    cfg = BASE_CONFIGS[model_key].copy()
    cfg.update(DEFAULT_HPARAMS)
    cfg.update(custom_hparams)

    if run_name is None:
        run_name = model_key
    cfg["key"] = run_name

    # Output directories
    if model_key == "xlmr":
        cfg["save_dir"]   = os.path.join(MODEL_ROOT, f"xlm-roberta-base_{run_name}")
        cfg["output_dir"] = os.path.join(TRAINING_ROOT, run_name)
        cfg["tok_dir"]    = os.path.join(TOKENIZER_ROOT, f"xlm-roberta-base_{run_name}")
    elif model_key == "mbert":
        cfg["save_dir"]   = os.path.join(MODEL_ROOT, f"mbert_{run_name}")
        cfg["output_dir"] = os.path.join(TRAINING_ROOT, run_name)
        cfg["tok_dir"]    = os.path.join(TOKENIZER_ROOT, f"mbert_{run_name}")
    elif model_key == "muril":
        cfg["save_dir"]   = os.path.join(MODEL_ROOT, f"muril_{run_name}")
        cfg["output_dir"] = os.path.join(TRAINING_ROOT, run_name)
        cfg["tok_dir"]    = os.path.join(TOKENIZER_ROOT, f"muril_{run_name}")
    else:
        raise ValueError(f"Unsupported model_key: {model_key}")

    if clean_old:
        for d in [cfg["save_dir"], cfg["output_dir"], cfg["tok_dir"]]:
            if os.path.exists(d):
                shutil.rmtree(d)

    # Class weights
    global class_weights
    if "class_weights" not in globals():
        if "train_df" not in globals():
            raise ValueError("train_df not defined. Run prepare_dataset() first.")
        classes = np.unique(train_df["label"].astype(int).to_numpy())
        cw = compute_class_weight("balanced", classes=classes,
                                  y=train_df["label"].astype(int).to_numpy())
        class_weights = torch.tensor(cw, dtype=torch.float)
        _log(f"Class weights: {class_weights.numpy().round(4)}")

    _log(f"\n{'#' * 80}")
    _log(f"  RUNNING: {cfg['label']} | Run: {run_name}")
    _log(f"  Model output : {cfg['save_dir']}")
    _log(f"  Checkpoints  : {cfg['output_dir']}")
    _log(f"  Tokenizer    : {cfg['tok_dir']}")
    _log(f"{'#' * 80}")

    result = run_pipeline_v6(
        config=cfg, train_df=train_df, val_df=val_df, test_df=test_df,
        class_weights=class_weights, report_to="none", n_trials=n_trials,
    )

    all_results[run_name] = result

    run_history.append({
        "run_name": run_name,
        "model": model_key,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "learning_rate": cfg["learning_rate"],
        "use_optuna": cfg["use_optuna"],
        "use_llrd": cfg["use_llrd"],
        "val_macro_f1": result.get("val_f1"),
        "test_macro_f1": result["test_macro_f1"],
        "test_accuracy": result["test_accuracy"],
        "mcc": result.get("mcc"),
        "balanced_accuracy": result.get("balanced_accuracy"),
        "mean_class_ece": result.get("mean_class_ece"),
        "gap_status": result.get("gap_status", ""),
    })

    _log(f"\n✅ {run_name} | Val F1: {result.get('val_f1', 0):.4f} "
         f"| Test F1: {result['test_macro_f1']:.4f}")
    return result


# ================================================================
# 11. COMPARISON
# ================================================================

def show_comparison():
    if not run_history:
        _log("No runs completed yet.")
        return None

    df = pd.DataFrame(run_history)
    cols = ["run_name", "model", "learning_rate", "use_optuna", "use_llrd",
            "val_macro_f1", "test_macro_f1", "balanced_accuracy",
            "mean_class_ece", "gap_status"]

    _log("\n" + "=" * 130)
    _log("RUN COMPARISON")
    _log("=" * 130)
    _log(df[cols].to_string(index=False))

    best_idx = df["val_macro_f1"].idxmax()
    best = df.iloc[best_idx]
    _log(f"\n🏆 Best (by Validation): {best['run_name']}")
    _log(f"   Val F1  = {best['val_macro_f1']:.4f}")
    _log(f"   Test F1 = {best['test_macro_f1']:.4f}")
    return df

In [60]:
run_model("muril", run_name="muril23", debug_mode=True)


################################################################################
  RUNNING: MuRIL | Run: muril23


  Model output : D:\Nepali sentiment\models\muril_muril23
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\muril23
  Tokenizer    : D:\Nepali sentiment\tokenizers\muril_muril23
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): MuRIL
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : False
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 1e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 31 tokens
     95.0% → 39 tokens
     97.0% → 45 tokens
     98.0% → 49 tokens
     99.0% → 55 tokens
     99.5% → 61 tokens
    100.0% → 92 tokens
    Current MAX_LEN = 256
    Truncation Rate = 0.00%
  Train: 8549 | Val: 1069 | Test: 1069

  Optuna disabled → Using fixed hyperparameters.
    learning_rate                      : 1e-05
    weight_decay                       : 0.15
    warmu

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

...

  Train Macro-F1 (1000 samples): 0.8175
  Val   Macro-F1 : 0.7815
  Gap            : +0.0360
  Status         : ✅ Healthy generalization

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.7437
  Balanced Accuracy : 0.7463
  Macro-F1          : 0.7429
  Weighted-F1       : 0.7411
  MCC               : 0.6173
  Macro ROC-AUC     : 0.8814
  Val→Test Gap      : +0.0386

[Bootstrap CI (n=1000)]
  accuracy            : 0.7441 [0.7194, 0.7699]
  macro_f1            : 0.7430 [0.7184, 0.7687]
  weighted_f1         : 0.7416 [0.7161, 0.7679]
  mcc                 : 0.6179 [0.5813, 0.6571]
  balanced_accuracy   : 0.7465 [0.7224, 0.7713]

[Calibration / ECE]
  Positive       : ECE = 0.1266 ⚠️
  Neutral        : ECE = 0.1259 ⚠️
  Negative       : ECE = 0.1330 ⚠️
  Mean Class ECE  : 0.1285

  GENERALIZATION ANALYSIS
  Train F1  : 0.8175
  Val   F1  : 0.7815
  Test  F1  : 0.7429
  Train→Val : +0.0360
  Val→Test  : +0.0386
  Status    : ✅ Healthy generalization

[Error Analysis]
  Saved 274 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\muril_muril23\model
  Tokenizer → D:\Nepali sentiment\tokenizers\muril_muril23
  Plots     → D:\Nepali sentiment\models\muril_muril23\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ muril23 | Val F1: 0.7815 | Test F1: 0.7429


{'label': 'MuRIL',
 'best_hparams': {'learning_rate': 1e-05,
  'weight_decay': 0.15,
  'warmup_ratio': 0.1,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.7436856875584659,
 'test_macro_f1': 0.7429234276850721,
 'test_weighted_f1': 0.7411040711296863,
 'balanced_accuracy': 0.7462703201508466,
 'mcc': 0.6173152294309686,
 'macro_roc_auc': np.float64(0.8814033159082125),
 'mean_class_ece': 0.1285085816564759,
 'per_class_f1': {'Positive': 0.841248303934871,
  'Neutral': 0.6481994459833795,
  'Negative': 0.7393225331369662},
 'roc_auc': {'Positive': 0.9461245896388822,
  'Neutral': 0.8004942403486924,
  'Negative': 0.897591117737063},
 'pr_auc': {'Positive': 0.9029588000793638,
  'Neutral': 0.6546832157068747,
  'Negative': 0.8247928400490386},
 'train_f1': 0.8174797048880992,
 'val_f1': 0.7814899640471841,
 'train_val_gap': 0.03598974084091511,
 'val_test_gap': 0.038566536362111936,
 'gap_status': '✅ Healthy generalization',
 'bootstrap_ci': {'accuracy': (0.7441028999064547,
 

In [21]:
run_model("mbert", run_name="mbert23", debug_mode=True)


################################################################################
  RUNNING: mBERT | Run: mbert23
  Model output : D:\Nepali sentiment\models\mbert_mbert23
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\mbert23
  Tokenizer    : D:\Nepali sentiment\tokenizers\mbert_mbert23
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): mBERT
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : False
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 1e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 61 tokens
     95.0% → 77 tokens
     97.0% → 87 tokens
     98.0% → 95 tokens
     99.0% → 107 tokens
     99.5% → 119 tokens
    100.0% → 194 tokens
    Current MAX_LEN = 256
    Truncation Rate = 0.00%
  Train: 8549 | Val: 1069 | Test: 1069

  Optuna disabled → Using fixed hy

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

  Train Macro-F1 (1000 samples): 0.8485
  Val   Macro-F1 : 0.7506
  Gap            : +0.0979
  Status         : ⚠️ Potential overfitting (Gap > 0.05)

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.7166
  Balanced Accuracy : 0.7186
  Macro-F1          : 0.7167
  Weighted-F1       : 0.7152
  MCC               : 0.5760
  Macro ROC-AUC     : 0.8788
  Val→Test Gap      : +0.0339

[Bootstrap CI (n=1000)]
  accuracy            : 0.7167 [0.6904, 0.7418]
  macro_f1            : 0.7164 [0.6901, 0.7422]
  weighted_f1         : 0.7153 [0.6887, 0.7413]
  mcc                 : 0.5763 [0.5366, 0.6142]
  balanced_accuracy   : 0.7186 [0.6929, 0.7437]

[Calibration / ECE]
  Positive       : ECE = 0.0592 🟠
  Neutral        : ECE = 0.0756 🟠
  Negative       : ECE = 0.0570 🟠
  Mean Class ECE  : 0.0639

  GENERALIZATION ANALYSIS
  Train F1  : 0.8485
  Val   F1  : 0.7506
  Test  F1  : 0.7167
  Train→Val : +0.0979
  Val→Test  : +0.0339
  Status    : ⚠️ Potential overfitting (Gap > 0.05)

[Error Analysis]
  Saved 303 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\mbert_mbert23\model
  Tokenizer → D:\Nepali sentiment\tokenizers\mbert_mbert23
  Plots     → D:\Nepali sentiment\models\mbert_mbert23\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ mbert23 | Val F1: 0.7506 | Test F1: 0.7167


{'label': 'mBERT',
 'best_hparams': {'learning_rate': 1e-05,
  'weight_decay': 0.15,
  'warmup_ratio': 0.1,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.7165575304022451,
 'test_macro_f1': 0.7166579050508375,
 'test_weighted_f1': 0.7151532968816229,
 'balanced_accuracy': 0.7185595764969679,
 'mcc': 0.5760229744106129,
 'macro_roc_auc': np.float64(0.8788396213904545),
 'mean_class_ece': 0.06391623358572983,
 'per_class_f1': {'Positive': 0.8,
  'Neutral': 0.6430517711171662,
  'Negative': 0.7069219440353461},
 'roc_auc': {'Positive': 0.9322523820962446,
  'Neutral': 0.8248871419676215,
  'Negative': 0.8793793401074972},
 'pr_auc': {'Positive': 0.8847567671530985,
  'Neutral': 0.6850155080355002,
  'Negative': 0.7970780767653839},
 'train_f1': 0.8485006448018284,
 'val_f1': 0.7505861398209448,
 'train_val_gap': 0.09791450498088361,
 'val_test_gap': 0.03392823477010731,
 'gap_status': '⚠️ Potential overfitting (Gap > 0.05)',
 'bootstrap_ci': {'accuracy': (0.7167184284377923,
 

In [23]:
run_model("xlmr", run_name="xlmr23", debug_mode=True)


################################################################################
  RUNNING: XLM-RoBERTa | Run: xlmr23
  Model output : D:\Nepali sentiment\models\xlm-roberta-base_xlmr23
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\xlmr23
  Tokenizer    : D:\Nepali sentiment\tokenizers\xlm-roberta-base_xlmr23
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): XLM-RoBERTa
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : False
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 1e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 36 tokens
     95.0% → 46 tokens
     97.0% → 53 tokens
     98.0% → 56 tokens
     99.0% → 63 tokens
     99.5% → 70 tokens
    100.0% → 106 tokens
    Current MAX_LEN = 256
    Truncation Rate = 0.00%
  Train: 8549 | Val: 1069 | Test: 1069

  Optu

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

  Train Macro-F1 (1000 samples): 0.8177
  Val   Macro-F1 : 0.7833
  Gap            : +0.0344
  Status         : ✅ Healthy generalization

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.7587
  Balanced Accuracy : 0.7605
  Macro-F1          : 0.7596
  Weighted-F1       : 0.7582
  MCC               : 0.6395
  Macro ROC-AUC     : 0.9071
  Val→Test Gap      : +0.0237

[Bootstrap CI (n=1000)]
  accuracy            : 0.7590 [0.7324, 0.7839]
  macro_f1            : 0.7597 [0.7327, 0.7853]
  weighted_f1         : 0.7586 [0.7311, 0.7839]
  mcc                 : 0.6400 [0.6000, 0.6783]
  balanced_accuracy   : 0.7607 [0.7355, 0.7858]

[Calibration / ECE]
  Positive       : ECE = 0.0448 ✅
  Neutral        : ECE = 0.0732 🟠
  Negative       : ECE = 0.0642 🟠
  Mean Class ECE  : 0.0607

  GENERALIZATION ANALYSIS
  Train F1  : 0.8177
  Val   F1  : 0.7833
  Test  F1  : 0.7596
  Train→Val : +0.0344
  Val→Test  : +0.0237
  Status    : ✅ Healthy generalization

[Error Analysis]
  Saved 258 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\xlm-roberta-base_xlmr23\model
  Tokenizer → D:\Nepali sentiment\tokenizers\xlm-roberta-base_xlmr23
  Plots     → D:\Nepali sentiment\models\xlm-roberta-base_xlmr23\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ xlmr23 | Val F1: 0.7833 | Test F1: 0.7596


{'label': 'XLM-RoBERTa',
 'best_hparams': {'learning_rate': 1e-05,
  'weight_decay': 0.15,
  'warmup_ratio': 0.1,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.7586529466791394,
 'test_macro_f1': 0.7596338824415064,
 'test_weighted_f1': 0.7582006748139527,
 'balanced_accuracy': 0.7604667705958268,
 'mcc': 0.6394573384143248,
 'macro_roc_auc': np.float64(0.9070615256565134),
 'mean_class_ece': 0.06073542840785797,
 'per_class_f1': {'Positive': 0.8361111111111111,
  'Neutral': 0.6827309236947792,
  'Negative': 0.7600596125186289},
 'roc_auc': {'Positive': 0.9540475618544318,
  'Neutral': 0.8523894769613948,
  'Negative': 0.9147475381537135},
 'pr_auc': {'Positive': 0.912042592493664,
  'Neutral': 0.7458357875002548,
  'Negative': 0.8510013108194618},
 'train_f1': 0.8176932792396459,
 'val_f1': 0.7833069844099256,
 'train_val_gap': 0.03438629482972022,
 'val_test_gap': 0.023673101968419208,
 'gap_status': '✅ Healthy generalization',
 'bootstrap_ci': {'accuracy': (0.75903461178

In [ ]:
# Fastest approach: Just use the best HPs directly, no searching
run_model(
    "xlmr",
    run_name="xlmr_fixed_best_hps",
    use_optuna=False,       # Optuna already done → use fixed HPs
    use_llrd=False,
    learning_rate=1.0953e-05,      # from Optuna: 1.0952662748632564e-05
    weight_decay=0.0365,           # from Optuna: 0.036503833523887946
    warmup_ratio=0.0938,           # from Optuna: 0.09382169728028272
    batch_size=32,                 # from Optuna: 32.0
)


################################################################################
  RUNNING: XLM-RoBERTa | Run: xlmr_fixed_best_hps
  Model output : D:\Nepali sentiment\models\xlm-roberta-base_xlmr_fixed_best_hps
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\xlmr_fixed_best_hps
  Tokenizer    : D:\Nepali sentiment\tokenizers\xlm-roberta-base_xlmr_fixed_best_hps
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): XLM-RoBERTa
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : False
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 1.0953e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 36 tokens
     95.0% → 46 tokens
     97.0% → 53 tokens
     98.0% → 56 tokens
     99.0% → 63 tokens
     99.5% → 70 tokens
    100.0% → 106 tokens
    Current MAX_LEN = 256
    Truncation Ra

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


  Accuracy          : 0.5323
  Balanced Accuracy : 0.5311
  Macro-F1          : 0.5240
  Weighted-F1       : 0.5237
  MCC               : 0.3094
  Macro ROC-AUC     : 0.7555
  Val→Test Gap      : +0.2691

[Bootstrap CI (n=1000)]
  accuracy            : 0.5322 [0.5042, 0.5631]
  macro_f1            : 0.5234 [0.4933, 0.5556]
  weighted_f1         : 0.5235 [0.4933, 0.5563]
  mcc                 : 0.3095 [0.2675, 0.3538]
  balanced_accuracy   : 0.5310 [0.5045, 0.5597]

[Calibration / ECE]
  Positive       : ECE = 0.1676 ⚠️
  Neutral        : ECE = 0.0765 🟠
  Negative       : ECE = 0.1339 ⚠️
  Mean Class ECE  : 0.1260

  GENERALIZATION ANALYSIS
  Train F1  : 0.8794
  Val   F1  : 0.7930
  Test  F1  : 0.5240
  Train→Val : +0.0863
  Val→Test  : +0.2691
  Status    : ⚠️ Potential overfitting (Gap > 0.05)

[Error Analysis]
  Saved 500 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\xlm-roberta-base_xlmr_fixed_best_hps\model
  Tokenizer → D:\Nepali sentiment\tokenizers\xlm-roberta-base_xlmr_fixed_best_hps
  Plots     → D:\Nepali sentiment\models\xlm-roberta-base_xlmr_fixed_best_hps\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ xlmr_fixed_best_hps | Val F1: 0.7930 | Test F1: 0.5240


{'label': 'XLM-RoBERTa',
 'best_hparams': {'learning_rate': 1.0953e-05,
  'weight_decay': 0.0365,
  'warmup_ratio': 0.0938,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.5322731524789522,
 'test_macro_f1': 0.523963776817045,
 'test_weighted_f1': 0.5237202622253095,
 'balanced_accuracy': 0.5310612578746,
 'mcc': 0.30940399003347063,
 'macro_roc_auc': np.float64(0.755479838812395),
 'mean_class_ece': 0.12600899875189586,
 'per_class_f1': {'Positive': 0.5067698259187621,
  'Neutral': 0.4573002754820937,
  'Negative': 0.6078212290502794},
 'roc_auc': {'Positive': 0.8002922153199178,
  'Neutral': 0.679313900996264,
  'Negative': 0.7868334001210034},
 'pr_auc': {'Positive': 0.6937117417589627,
  'Neutral': 0.5191097606712296,
  'Negative': 0.6784077140144009},
 'train_f1': 0.8793595546786922,
 'val_f1': 0.7930451695700458,
 'train_val_gap': 0.08631438510864642,
 'val_test_gap': 0.2690813927530008,
 'gap_status': '⚠️ Potential overfitting (Gap > 0.05)',
 'bootstrap_ci': {'accuracy

: 

In [21]:
run_model(
    "muril",
    use_optuna=True,
    n_trials=5,
    search_epochs=5,
    use_llrd=True,
    layer_decay=0.85,
)


################################################################################
  RUNNING: MuRIL | Run: muril
  Model output : D:\Nepali sentiment\models\muril_muril
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\muril
  Tokenizer    : D:\Nepali sentiment\tokenizers\muril_muril
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): MuRIL
  Seed            : 42 (single)
  Optuna          : True
  LLRD            : True
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 2e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

  Train Macro-F1 (1000 samples): 0.8207
  Val   Macro-F1 : 0.7701
  Gap            : +0.0507
  Status         : ⚠️ Potential overfitting (Gap > 0.05)

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.5126
  Balanced Accuracy : 0.5098
  Macro-F1          : 0.4990
  Weighted-F1       : 0.4998
  MCC               : 0.2799
  Macro ROC-AUC     : 0.7071
  Val→Test Gap      : +0.2710

[Bootstrap CI (n=1000)]
  accuracy            : 0.5128 [0.4827, 0.5426]
  macro_f1            : 0.4989 [0.4657, 0.5289]
  weighted_f1         : 0.5000 [0.4657, 0.5306]
  mcc                 : 0.2803 [0.2344, 0.3240]
  balanced_accuracy   : 0.5101 [0.4801, 0.5381]

[Calibration / ECE]
  Positive       : ECE = 0.1573 ⚠️
  Neutral        : ECE = 0.0880 🟠
  Negative       : ECE = 0.0771 🟠
  Mean Class ECE  : 0.1074

  GENERALIZATION ANALYSIS
  Train F1  : 0.8207
  Val   F1  : 0.7701
  Test  F1  : 0.4990
  Train→Val : +0.0507
  Val→Test  : +0.2710
  Status    : ⚠️ Potential overfitting (Gap > 0.05)

[Error Analysis]
  Saved 521 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\muril_muril\model
  Tokenizer → D:\Nepali sentiment\tokenizers\muril_muril
  Plots     → D:\Nepali sentiment\models\muril_muril\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ muril | Val F1: 0.7701 | Test F1: 0.4990


{'label': 'MuRIL',
 'best_hparams': {'learning_rate': 1.0952662748632564e-05,
  'weight_decay': 0.036503833523887946,
  'warmup_ratio': 0.09382169728028272,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.5126286248830683,
 'test_macro_f1': 0.49903928330553615,
 'test_weighted_f1': 0.4997914578720189,
 'balanced_accuracy': 0.5098354073484913,
 'mcc': 0.2799081850609773,
 'macro_roc_auc': np.float64(0.7071498406024802),
 'mean_class_ece': 0.10743394710116567,
 'per_class_f1': {'Positive': 0.4197002141327623,
  'Neutral': 0.4778972520908005,
  'Negative': 0.5995203836930456},
 'roc_auc': {'Positive': 0.7427745664739884,
  'Neutral': 0.6275120641344956,
  'Negative': 0.7511628911989565},
 'pr_auc': {'Positive': 0.6375165569672671,
  'Neutral': 0.4646891474168213,
  'Negative': 0.6058495120482558},
 'train_f1': 0.8207478980341049,
 'val_f1': 0.7700544918297076,
 'train_val_gap': 0.05069340620439722,
 'val_test_gap': 0.2710152085241715,
 'gap_status': '⚠️ Potential overfitting (Ga

In [ ]:
# Run mBERT with your specific best hyperparameters
run_model(
    "mbert",
    run_name="mbert_best_hps",      # Custom name to identify this run
    clean_old=True,                 # Clears previous mBERT runs to save disk space
    use_optuna=False,               # Disable Optuna search since we have fixed HPs
    use_llrd=True,                  # Keep Layer-wise Learning Rate Decay enabled (set to False if you want to test without it)
    learning_rate=1.0952662748632564e-05,
    weight_decay=0.036503833523887946,
    warmup_ratio=0.09382169728028272,
    batch_size=32                   # This maps to per_device_train_batch_size in the pipeline
)


################################################################################
  RUNNING: mBERT | Run: mbert_best_hps
  Model output : D:\FinalYear_Sentiment_Training\English_Sentiment_Models\models\mbert_mbert_best_hps
  Checkpoints  : D:\FinalYear_Sentiment_Training\English_Sentiment_Models\training_checkpoints\mbert_best_hps
  Tokenizer    : D:\FinalYear_Sentiment_Training\English_Sentiment_Models\tokenizers\mbert_mbert_best_hps
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): mBERT
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : True
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 1.0952662748632564e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 41 tokens
     95.0% → 50 tokens
     97.0% → 57 tokens
     98.0% → 61 tokens
     99.0% → 68 tokens
     99.5% → 76 tokens

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

  Train Macro-F1 (1000 samples): 0.9150
  Val   Macro-F1 : 0.7894
  Gap            : +0.1257
  Status         : ⚠️ Potential overfitting (Gap > 0.05)

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.7811
  Balanced Accuracy : 0.7816
  Macro-F1          : 0.7804
  Weighted-F1       : 0.7798
  MCC               : 0.6720
  Macro ROC-AUC     : 0.9151
  Val→Test Gap      : +0.0090

[Bootstrap CI (n=1000)]
  accuracy            : 0.7811 [0.7558, 0.8064]
  macro_f1            : 0.7801 [0.7548, 0.8052]
  weighted_f1         : 0.7798 [0.7546, 0.8052]
  mcc                 : 0.6720 [0.6340, 0.7103]
  balanced_accuracy   : 0.7816 [0.7572, 0.8063]

[Calibration / ECE]
  Positive       : ECE = 0.0391 ✅
  Neutral        : ECE = 0.0530 🟠
  Negative       : ECE = 0.0290 ✅
  Mean Class ECE  : 0.0404

  GENERALIZATION ANALYSIS
  Train F1  : 0.9150
  Val   F1  : 0.7894
  Test  F1  : 0.7804
  Train→Val : +0.1257
  Val→Test  : +0.0090
  Status    : ⚠️ Potential overfitting (Gap > 0.05)

[Error Analysis]
  Saved 234 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\FinalYear_Sentiment_Training\English_Sentiment_Models\models\mbert_mbert_best_hps\model
  Tokenizer → D:\FinalYear_Sentiment_Training\English_Sentiment_Models\tokenizers\mbert_mbert_best_hps
  Plots     → D:\FinalYear_Sentiment_Training\English_Sentiment_Models\models\mbert_mbert_best_hps\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ mbert_best_hps | Val F1: 0.7894 | Test F1: 0.7804


{'label': 'mBERT',
 'best_hparams': {'learning_rate': 1.0952662748632564e-05,
  'weight_decay': 0.036503833523887946,
  'warmup_ratio': 0.09382169728028272,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.7811038353601497,
 'test_macro_f1': 0.7803611398447545,
 'test_weighted_f1': 0.7798109198131308,
 'balanced_accuracy': 0.7816253924141243,
 'mcc': 0.6719933058570462,
 'macro_roc_auc': np.float64(0.9151246681909947),
 'mean_class_ece': 0.040384989773136155,
 'per_class_f1': {'Positive': 0.8534850640113798,
  'Neutral': 0.6897546897546898,
  'Negative': 0.7978436657681941},
 'roc_auc': {'Positive': 0.9561246418338109,
  'Neutral': 0.8587008619538145,
  'Negative': 0.9305485007853588},
 'pr_auc': {'Positive': 0.9222389036091942,
  'Neutral': 0.7487475252046336,
  'Negative': 0.8696338432716411},
 'train_f1': 0.9150092708464577,
 'val_f1': 0.7893549024332783,
 'train_val_gap': 0.12565436841317934,
 'val_test_gap': 0.008993762588523802,
 'gap_status': '⚠️ Potential overfitting (

In [22]:
run_model(
    "mbert",
    use_optuna=True,
    n_trials=5,
    search_epochs=5,
    use_llrd=True,
    layer_decay=0.85,
)


################################################################################
  RUNNING: mBERT | Run: mbert
  Model output : D:\Nepali sentiment\models\mbert_mbert
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\mbert
  Tokenizer    : D:\Nepali sentiment\tokenizers\mbert_mbert
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): mBERT
  Seed            : 42 (single)
  Optuna          : True
  LLRD            : True
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 2e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  Train Macro-F1 (1000 samples): 0.8221
  Val   Macro-F1 : 0.7494
  Gap            : +0.0728
  Status         : ⚠️ Potential overfitting (Gap > 0.05)

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.4818
  Balanced Accuracy : 0.4793
  Macro-F1          : 0.4590
  Weighted-F1       : 0.4601
  MCC               : 0.2336
  Macro ROC-AUC     : 0.6982
  Val→Test Gap      : +0.2904

[Bootstrap CI (n=1000)]
  accuracy            : 0.4816 [0.4509, 0.5117]
  macro_f1            : 0.4586 [0.4286, 0.4896]
  weighted_f1         : 0.4599 [0.4281, 0.4917]
  mcc                 : 0.2336 [0.1889, 0.2785]
  balanced_accuracy   : 0.4793 [0.4524, 0.5083]

[Calibration / ECE]
  Positive       : ECE = 0.1617 ⚠️
  Neutral        : ECE = 0.0584 🟠
  Negative       : ECE = 0.1334 ⚠️
  Mean Class ECE  : 0.1178

  GENERALIZATION ANALYSIS
  Train F1  : 0.8221
  Val   F1  : 0.7494
  Test  F1  : 0.4590
  Train→Val : +0.0728
  Val→Test  : +0.2904
  Status    : ⚠️ Potential overfitting (Gap > 0.05)

[Error Analysis]
  Saved 554 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\mbert_mbert\model
  Tokenizer → D:\Nepali sentiment\tokenizers\mbert_mbert
  Plots     → D:\Nepali sentiment\models\mbert_mbert\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ mbert | Val F1: 0.7494 | Test F1: 0.4590


{'label': 'mBERT',
 'best_hparams': {'learning_rate': 1.0952662748632564e-05,
  'weight_decay': 0.036503833523887946,
  'warmup_ratio': 0.09382169728028272,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.48175865294667913,
 'test_macro_f1': 0.4590033183227021,
 'test_weighted_f1': 0.460072584139464,
 'balanced_accuracy': 0.47933218938469335,
 'mcc': 0.23362427646578765,
 'macro_roc_auc': np.float64(0.6982195722449376),
 'mean_class_ece': 0.11784645140700745,
 'per_class_f1': {'Positive': 0.35490605427974947,
  'Neutral': 0.4438430311231394,
  'Negative': 0.5782608695652174},
 'roc_auc': {'Positive': 0.7355591266319685,
  'Neutral': 0.6341901463262765,
  'Negative': 0.7249094437765677},
 'pr_auc': {'Positive': 0.5797579507734063,
  'Neutral': 0.4877106553824858,
  'Negative': 0.5664016939744188},
 'train_f1': 0.8221199059280296,
 'val_f1': 0.7493672275685673,
 'train_val_gap': 0.07275267835946231,
 'val_test_gap': 0.29036390924586525,
 'gap_status': '⚠️ Potential overfitting 

In [28]:
# Save original epochs value
original_epochs = EPOCHS

# Set epochs to 5 for this run
EPOCHS = 5

# Run XLM-R with Optuna-found HPs and 5 epochs
run_model(
    "xlmr",
    run_name="xlmr_fixed_best_hps_5ep",
    use_optuna=False,       # Optuna already done → use fixed HPs
    use_llrd=False,
    learning_rate=1.0952662748632564e-05,      # from Optuna: 1.0952662748632564e-05
    weight_decay=0.036503833523887946,           # from Optuna: 0.036503833523887946
    warmup_ratio=0.09382169728028272
,           # from Optuna: 0.09382169728028272
    batch_size=32,                 # from Optuna: 32.0
)

# Restore original epochs for future runs
EPOCHS = original_epochs


################################################################################
  RUNNING: XLM-RoBERTa | Run: xlmr_fixed_best_hps_5ep
  Model output : D:\Nepali sentiment\models\xlm-roberta-base_xlmr_fixed_best_hps_5ep
  Checkpoints  : D:\Nepali sentiment\training_checkpoints\xlmr_fixed_best_hps_5ep
  Tokenizer    : D:\Nepali sentiment\tokenizers\xlm-roberta-base_xlmr_fixed_best_hps_5ep
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): XLM-RoBERTa
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : False
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 1.0952662748632564e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 36 tokens
     95.0% → 46 tokens
     97.0% → 53 tokens
     98.0% → 56 tokens
     99.0% → 63 tokens
     99.5% → 70 tokens
    100.0% → 106 tokens
    Current MAX

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Micro F1,Weighted F1
0.154822,0.206196,5,0.789523,0.788949,0.789523,0.787681


  Train Macro-F1 (1000 samples): 0.8190
  Val   Macro-F1 : 0.7889
  Gap            : +0.0300
  Status         : ✅ Healthy generalization

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.5285
  Balanced Accuracy : 0.5269
  Macro-F1          : 0.5226
  Weighted-F1       : 0.5225
  MCC               : 0.3033
  Macro ROC-AUC     : 0.7495
  Val→Test Gap      : +0.2664

[Bootstrap CI (n=1000)]
  accuracy            : 0.5283 [0.5005, 0.5575]
  macro_f1            : 0.5217 [0.4942, 0.5515]
  weighted_f1         : 0.5220 [0.4939, 0.5521]
  mcc                 : 0.3031 [0.2610, 0.3463]
  balanced_accuracy   : 0.5268 [0.4993, 0.5550]

[Calibration / ECE]
  Positive       : ECE = 0.1627 ⚠️
  Neutral        : ECE = 0.0836 🟠
  Negative       : ECE = 0.1151 ⚠️
  Mean Class ECE  : 0.1204

  GENERALIZATION ANALYSIS
  Train F1  : 0.8190
  Val   F1  : 0.7889
  Test  F1  : 0.5226
  Train→Val : +0.0300
  Val→Test  : +0.2664
  Status    : ✅ Healthy generalization

[Error Analysis]
  Saved 504 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\Nepali sentiment\models\xlm-roberta-base_xlmr_fixed_best_hps_5ep\model
  Tokenizer → D:\Nepali sentiment\tokenizers\xlm-roberta-base_xlmr_fixed_best_hps_5ep
  Plots     → D:\Nepali sentiment\models\xlm-roberta-base_xlmr_fixed_best_hps_5ep\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ xlmr_fixed_best_hps_5ep | Val F1: 0.7889 | Test F1: 0.5226


In [ ]:
# Configuration for XLM-RoBERTa with best hyperparameters
xlmr_config = {
    'key': 'xlmr',
    'label': 'XLM-RoBERTa',
    'checkpoint': 'xlm-roberta-base',
    'save_dir': './results_xlmr',
    'output_dir': './runs/xlmr',
    'tok_dir': './tokenizers_xlmr',
    
    # Disable Optuna since we're using fixed best hyperparameters
    'use_optuna': False,
    
    # Optional settings
    'use_llrd': False,           # Layer-wise learning rate decay
    'layer_decay': 0.9,
    'use_custom_model': False,  # Use standard AutoModel
    
    # Best hyperparameters found
    'learning_rate': 4.33e-06,
    'weight_decay': 0.190635718,
    'warmup_ratio': 0.159799091,
    'batch_size': 16,
}

# Run the pipeline
result = run_training_pipeline_optimized(
    config=xlmr_config,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_weights=class_weights,
    report_to='none',
)

print(f"\n✅ XLM-RoBERTa Training Complete!")
print(f"Test Macro-F1: {result['test_macro_f1']:.4f}")
print(f"Test Accuracy: {result['test_accuracy']:.4f}")

In [ ]:
# import sys
# import io
# import faulthandler

# def _watchdog_arm(minutes):
#     if minutes and minutes > 0:
#         try:
#             # Jupyter's sys.stderr is an OutStream and lacks fileno().
#             # We attempt to use the original OS-level stderr stream instead.
#             if hasattr(sys, '__stderr__') and sys.__stderr__ is not None:
#                 faulthandler.dump_traceback_later(
#                     minutes * 60, 
#                     repeat=True, 
#                     file=sys.__stderr__
#                 )
#             else:
#                 raise io.UnsupportedOperation("No original stderr available")
                
#             _log(f"  ⏱️  Watchdog armed: stack dump every {minutes} min if stalled.")
            
#         except (io.UnsupportedOperation, AttributeError, ValueError, OSError):
#             # Fallback for environments where faulthandler cannot get a valid file descriptor
#             _log(f"  ⏱️  Watchdog disabled: Jupyter/IPython environment does not support faulthandler fileno().")

In [ ]:
run_model("xlmr", run_name="smoke", debug_mode=True)


################################################################################
  RUNNING: XLM-RoBERTa | Run: smoke
  Model output : D:\FinalYear_Sentiment_Training\English_Sentiment_Models\models\xlm-roberta-base_smoke
  Checkpoints  : D:\FinalYear_Sentiment_Training\English_Sentiment_Models\training_checkpoints\smoke
  Tokenizer    : D:\FinalYear_Sentiment_Training\English_Sentiment_Models\tokenizers\xlm-roberta-base_smoke
################################################################################

  PIPELINE v6.1 (SPEED-OPTIMIZED): XLM-RoBERTa
  Seed            : 42 (single)
  Optuna          : False
  LLRD            : False
  Grad Ckpt       : False
  Batch/Accum     : 32/1
  Learning Rate   : 2e-05

[1] Strict Data QA
  ✅ Passed strict QA

[2] Tokenization & Length Analysis

  [Token Length Distribution]  (sampled 5000)
     90.0% → 40 tokens
     95.0% → 49 tokens
     97.0% → 56 tokens
     98.0% → 59 tokens
     99.0% → 67 tokens
     99.5% → 73 tokens
    100.0% → 113 

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

  Train Macro-F1 (1000 samples): 0.8459
  Val   Macro-F1 : 0.8003
  Gap            : +0.0456
  Status         : ✅ Healthy generalization

--------------------------------------------------------------------------------
  INDEPENDENT TEST EVALUATION
--------------------------------------------------------------------------------



  Accuracy          : 0.8026
  Balanced Accuracy : 0.8034
  Macro-F1          : 0.8022
  Weighted-F1       : 0.8016
  MCC               : 0.7042
  Macro ROC-AUC     : 0.9323
  Val→Test Gap      : -0.0019

[Bootstrap CI (n=1000)]
  accuracy            : 0.8025 [0.7774, 0.8279]
  macro_f1            : 0.8019 [0.7772, 0.8271]
  weighted_f1         : 0.8015 [0.7763, 0.8272]
  mcc                 : 0.7041 [0.6665, 0.7426]
  balanced_accuracy   : 0.8034 [0.7792, 0.8282]

[Calibration / ECE]
  Positive       : ECE = 0.0572 🟠
  Neutral        : ECE = 0.1284 ⚠️
  Negative       : ECE = 0.0944 🟠
  Mean Class ECE  : 0.0933

  GENERALIZATION ANALYSIS
  Train F1  : 0.8459
  Val   F1  : 0.8003
  Test  F1  : 0.8022
  Train→Val : +0.0456
  Val→Test  : -0.0019
  Status    : ✅ Healthy generalization

[Error Analysis]
  Saved 211 errors → error_analysis.csv

[Saving Plots]

[Saving Artifacts]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Model     → D:\FinalYear_Sentiment_Training\English_Sentiment_Models\models\xlm-roberta-base_smoke\model
  Tokenizer → D:\FinalYear_Sentiment_Training\English_Sentiment_Models\tokenizers\xlm-roberta-base_smoke
  Plots     → D:\FinalYear_Sentiment_Training\English_Sentiment_Models\models\xlm-roberta-base_smoke\plots

  PIPELINE v6.1 COMPLETED SUCCESSFULLY

✅ smoke | Val F1: 0.8003 | Test F1: 0.8022


{'label': 'XLM-RoBERTa',
 'best_hparams': {'learning_rate': 2e-05,
  'weight_decay': 0.15,
  'warmup_ratio': 0.1,
  'per_device_train_batch_size': 32},
 'test_accuracy': 0.8026192703461179,
 'test_macro_f1': 0.802214299792737,
 'test_weighted_f1': 0.8016172864571386,
 'balanced_accuracy': 0.8034488370928236,
 'mcc': 0.7042253545985974,
 'macro_roc_auc': np.float64(0.9322666840748416),
 'mean_class_ece': 0.09330721417895575,
 'per_class_f1': {'Positive': 0.8758716875871687,
  'Neutral': 0.7223796033994334,
  'Negative': 0.8083916083916084},
 'roc_auc': {'Positive': 0.9686564788283984,
  'Neutral': 0.8847401959628818,
  'Negative': 0.9434033774332445},
 'pr_auc': {'Positive': 0.9443405674989013,
  'Neutral': 0.7720850188758724,
  'Negative': 0.9049747602118722},
 'train_f1': 0.8458756454143446,
 'val_f1': 0.8002656992678924,
 'train_val_gap': 0.045609946146452196,
 'val_test_gap': -0.0019486005248445792,
 'gap_status': '✅ Healthy generalization',
 'bootstrap_ci': {'accuracy': (0.80250327

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

if 'class_weights' not in globals():
    classes = np.unique(train_df['label'].astype(int).to_numpy())
    cw = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=train_df['label'].astype(int).to_numpy(),
    )
    class_weights = torch.tensor(cw, dtype=torch.float)
    print(f"Class weights (Pos / Neu / Neg): {class_weights.numpy().round(4)}")


## 11. Disk Space Cleanup (safety net)

`run_sentiment_pipeline` already cleans up its own local checkpoint dir after each run; this cell is just a safety net in case a run was interrupted.

In [ ]:
import glob

for directory in glob.glob('./results_*'):
    if os.path.exists(directory):
        print(f"Removing {directory} to free up space...")
        shutil.rmtree(directory)

print("\nDisk cleanup complete.")


Disk cleanup complete.


## 12. Comparison Table Against Thapa et al. (2023)

Built dynamically from the `results` dict — every entry reflects a model that was actually trained and evaluated in this notebook, not a remembered/hard-coded score. Run only the models you care about above; this table adapts to whatever is in `results`.

In [23]:
PAPER_RESULTS = {
    'mBERT (paper)' : 0.61,
    'MuRIL (paper)' : 0.65,
    'XLM-R (paper)' : 0.67,
}

OUR_RESULTS = {
    f"{cfg['label']} (ours)": all_results[cfg['key']]['macro_f1']
    for cfg in MODEL_CONFIGS if cfg['key'] in all_results
}

all_scores = {**PAPER_RESULTS, **OUR_RESULTS}

print("\n📊 Macro-F1 Comparison")
print("-" * 50)
for name, score in sorted(all_scores.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(score * 40)
    print(f"{name:<24} {score:.4f}  {bar}")

summary_df = pd.DataFrame([
    {
        'Model'          : cfg['label'],
        'Checkpoint'     : cfg['checkpoint'],
        'Macro F1 (ours)': all_results[cfg['key']]['macro_f1'],
        'Accuracy (ours)': all_results[cfg['key']]['accuracy'],
    }
    for cfg in MODEL_CONFIGS if cfg['key'] in all_results
]).sort_values('Macro F1 (ours)', ascending=False).reset_index(drop=True)

summary_df

KeyError: 'macro_f1'

```markdown
## 12.1. Visualizing Training & Validation Loss

The following function extracts the loss history from each model's `results` entry and plots them to check for convergence or overfitting.
```

In [24]:
plot_labels = list(all_scores.keys())
plot_scores = list(all_scores.values())
plot_colors = ['#4C72B0' if '(ours)' in l else '#B0B0B0' for l in plot_labels]

order = np.argsort(plot_scores)[::-1]
plot_labels = [plot_labels[i] for i in order]
plot_scores = [plot_scores[i] for i in order]
plot_colors = [plot_colors[i] for i in order]

plt.figure(figsize=(8, 5))
plt.barh(plot_labels, plot_scores, color=plot_colors)
plt.xlabel('Macro F1')
plt.title('Macro-F1: Paper Baselines vs. Ours')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('macro_f1_comparison.png', dpi=150)
plt.show()

NameError: name 'all_scores' is not defined

In [25]:
def plot_loss_curves(results_dict, smooth=True, window=5):
    if not results_dict:
        print("No results found. Please train at least one model first.")
        return

    num_models = len(results_dict)
    fig, axes = plt.subplots(1, num_models, figsize=(6 * num_models, 5), sharey=True)
    if num_models == 1: axes = [axes]

    for i, (model_key, data) in enumerate(results_dict.items()):
        history = data['history']

        train_loss = [log['loss'] for log in history if 'loss' in log]
        train_steps = [log['epoch'] for log in history if 'loss' in log]
        val_loss = [log['eval_loss'] for log in history if 'eval_loss' in log]
        val_epochs = [log['epoch'] for log in history if 'eval_loss' in log]

        ax = axes[i]

        # Original noisy training loss in light color
        ax.plot(train_steps, train_loss, color='blue', alpha=0.2, label='Raw Train Loss')

        if smooth and len(train_loss) > window:
            # Apply moving average to smooth out fluctuations
            smoothed_loss = np.convolve(train_loss, np.ones(window)/window, mode='valid')
            ax.plot(train_steps[window-1:], smoothed_loss, color='blue', linewidth=2, label='Smoothed Train')

        ax.plot(val_epochs, val_loss, label='Validation Loss', color='red', marker='o', linewidth=2)

        ax.set_title(f'Loss Trends: {model_key}')
        ax.set_xlabel('Epochs')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# Re-run with smoothing to see the real trend
plot_loss_curves(results, smooth=True, window=10)

NameError: name 'results' is not defined

## 13. Inference Helper

Quick function to run inference on new text using any model that's been trained above. Reloads the model/tokenizer from its saved Drive directory (since `run_sentiment_pipeline` frees GPU memory after each run by default).

In [26]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_trained(model_key: str):
    """Reload a trained model + tokenizer from Drive using its entry in `results`."""
    r = results[model_key]
    tok = AutoTokenizer.from_pretrained(r['tok_save'])
    mdl = AutoModelForSequenceClassification.from_pretrained(r['save_dir']).to(device)
    return mdl, tok

def predict_sentiment(texts, model, tokenizer, device='cuda', batch_size=32):
    """
    Predict sentiment for a list of texts.
    Returns list of (label_str, confidence) tuples.
    """
    model.eval()
    results_out = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(
            batch_texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits

        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)

        for pred, prob in zip(preds, probs):
            results_out.append((ID2LABEL[pred], round(float(prob[pred]), 4)))

    return results_out


# ── Quick smoke test — uses whichever model in `results` scored highest ──────
if results:
    best_key = max(results, key=lambda k: results[k]['macro_f1'])
    demo_model, demo_tokenizer = load_trained(best_key)

    sample_texts  = val_df['input_text'].head(5).tolist()
    sample_labels = val_df['sentiment_raw'].head(5).tolist()

    predictions = predict_sentiment(sample_texts, demo_model, demo_tokenizer, device=device)
    print(f"Using best model so far: {best_key} (macro F1 = {results[best_key]['macro_f1']:.4f})\n")
    print(f"{'Text':<60} {'True':>10} {'Pred':>10} {'Conf':>6}")
    print("-" * 90)
    for text, true_lbl, (pred_lbl, conf) in zip(sample_texts, sample_labels, predictions):
        match = '✓' if true_lbl == pred_lbl else '✗'
        print(f"{str(text)[:58]:<60} {true_lbl:>10} {pred_lbl:>10} {conf:>6.3f} {match}")
else:
    print("No models trained yet — run one of the `run_sentiment_pipeline(...)` cells above first.")

NameError: name 'results' is not defined